In [ ]:
# del pip_install
try:
  if pip_install:
    pass
  else:
    raise NameError
except NameError as err:
  !pip install datasets==3.0.0
  # !pip install datasets==3.0.0 --force-reinstall --no-cache-dir --upgrade
  # !pip install timm
  pip_install=True

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 474.3/474.3 kB 23.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.6/177.6 kB 17.6 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2025.3.0
    Uninstalling fsspec-2025.3.0:
      Successfully uninstalled fsspec-2025.3.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2024.6.1 which is incompatible.


In [ ]:
# color_print.py

class ColorPrint:
    HEADER = '\033[95m'
    BLUE = '\033[94m'
    CYAN = '\033[96m'
    GREEN = '\033[92m'
    YELLOW = '\033[93m'
    RED = '\033[91m'
    BOLD = '\033[1m'
    UNDERLINE = '\033[4m'
    RESET = '\033[0m'

    @staticmethod
    def red(text):
        print(f"{ColorPrint.RED}{text}{ColorPrint.RESET}")

    @staticmethod
    def green(*text):
        print(f"{ColorPrint.GREEN}{text}{ColorPrint.RESET}")

    @staticmethod
    def yellow(*text):
        print(f"{ColorPrint.YELLOW}{text}{ColorPrint.RESET}")

    @staticmethod
    def blue(*text):
        print(f"{ColorPrint.BLUE}{text}{ColorPrint.RESET}")

    @staticmethod
    def cyan(*text):
        print(f"{ColorPrint.CYAN}{text}{ColorPrint.RESET}")

    @staticmethod
    def header(*text):
        print(f"{ColorPrint.HEADER}{text}{ColorPrint.RESET}")

    @staticmethod
    def bold(*text):
        print(f"{ColorPrint.BOLD}{text}{ColorPrint.RESET}")

    @staticmethod
    def underline(*text):
        print(f"{ColorPrint.UNDERLINE}{text}{ColorPrint.RESET}")

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from datasets import load_dataset
from PIL import Image
from torchvision import transforms
import urllib

import math
import os
import io

import numpy as np
import torch
import torch.nn.functional as F
from torch import nn
from timm.models.layers import DropPath, to_2tuple, trunc_normal_

try:
    import os, sys

    kernel_path = os.path.abspath(os.path.join('..'))
    sys.path.append(kernel_path)
    from kernels.window_process.window_process import WindowProcess, WindowProcessReverse

except:
    WindowProcess = None
    WindowProcessReverse = None
    print("[Warning] Fused window process have not been installed. Please refer to get_started.md for installation.")


[Warning] Fused window process have not been installed. Please refer to get_started.md for installation.


/usr/local/lib/python3.12/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


## quantization

### Base Observer

In [ ]:
class ObserverBase(nn.Module):
    def __init__(self, dtype=torch.qint8, qscheme=torch.per_tensor_affine):
        """
        ObserverBase class is a base class for observers in PyTorch quantization.

        Args:
            dtype (torch.dtype): Data type for quantization, typically `torch.qint8` or `torch.quint8`.
            qscheme (torch.qscheme): Quantization scheme. For example, `torch.per_tensor_affine`.
        """
        super(ObserverBase, self).__init__()
        self.dtype = dtype
        self.qscheme = qscheme
        self.is_symmetric = qscheme in [torch.per_tensor_symmetric, torch.per_channel_symmetric]

    def forward(self, x):
        """
        Placeholder for the forward method that will process the input data (activations or weights).
        To be overridden in derived classes.

        Args:
            x (torch.Tensor): The input tensor to observe.
        """
        raise NotImplementedError("ObserverBase.forward must be implemented in derived classes")

    def calculate_qparams(self):
        """
        Placeholder for the calculate_qparams method. It should calculate the quantization parameters
        (scale and zero point) based on the statistics collected in the forward pass.
        """
        raise NotImplementedError("ObserverBase.calculate_qparams must be implemented in derived classes")

### MinMaxObserver

In [ ]:

class MinMaxObserver(ObserverBase):
    def __init__(self,
                 dtype=torch.qint8,
                 qscheme=torch.per_tensor_affine,
                 nof_bits=8,
                 is_calibrate=False,
                 name='base'):
        """
        MinMaxObserver class is used to record the minimum and maximum values of the input activations.
        These values will be used to compute the scale and zero-point for quantization.

        Args:
            dtype (torch.dtype): The data type for quantization, e.g., torch.qint8.
            qscheme (torch.qscheme): The quantization scheme, e.g., torch.per_tensor_affine.
            nof_bits (int): Number of bits for quantization.
            is_calibrate (bool): Flag indicating whether to calculate the scale and zero-point during forwarding.
        """
        super(MinMaxObserver, self).__init__()
        self.dtype = dtype
        self.qscheme = qscheme
        self.nof_bits = nof_bits
        self.is_calibrate = is_calibrate
        self.min_val = torch.tensor(float('inf'))  # Initialize min with infinity
        self.max_val = torch.tensor(float('-inf'))  # Initialize max with -infinity
        self.zero_point = None
        self.scale = None
        self.name = name
        self.ez = None
        self.nml = None
        self.sz  = None

    def forward(self, x):
        """
        This method updates the minimum and maximum values by comparing the current min and max
        with those in the input tensor. If is_calibrate is True, it calculates the scale and zero-point.

        Args:
            x (torch.Tensor): The input tensor to observe.
        """
        global_min = x.min()
        global_max = x.max()

        if global_min < self.min_val:
            self.min_val = global_min
        if global_max > self.max_val:
            self.max_val = global_max

        # Calculate scale and zero-point during forwarding if is_calibrate is True
        self.calculate_qparams()
        self.ez = x.element_size()
        self.nml = x.numel()
        self.sz = x.size()

        return x

    def calculate_qparams(self):
        """
        This method computes the scale and zero-point for quantization based on the min and max values
        observed during the forward pass.

        Returns:
            scale (torch.Tensor): The quantization scale.
            zero_point (torch.Tensor): The quantization zero-point.
        """
        if self.min_val == self.max_val:
            # To handle cases where min == max to avoid division by zero
            self.scale = torch.tensor([1.0], dtype=torch.float32)
            self.zero_point = torch.tensor([0], dtype=torch.int32)
        else:
            # Calculate scale and zero-point for qint8 quantization
            self.scale = (self.max_val - self.min_val) / (2 ** self.nof_bits - 1)
            if self.qscheme == torch.per_tensor_affine:
                self.zero_point = torch.round(-self.min_val / self.scale).clamp(0, 2 ** self.nof_bits - 1)
            else:
                self.zero_point = 0
        return self.scale, self.zero_point

    def quantizer(self, x, scale=None):
        """
        Quantizes the input tensor using the calculated scale and zero-point.

        Args:
            x (torch.Tensor): The input tensor to quantize.

        Returns:
            torch.Tensor: The quantized tensor.
        """
        # if self.is_calibrate:
        if self.scale is None or self.zero_point is None:
            raise ValueError(f"Scale and zero-point must be calculated before quantizing [{self.name}].")

        # Quantize the input
        if self.qscheme == torch.per_tensor_affine:
            x_q = (x / (self.scale) + self.zero_point).round().clamp(0, 2 ** self.nof_bits - 1)
        else:
            x_q = (x / (self.scale) + self.zero_point).round().clamp(-2 ** (self.nof_bits-1), 2 ** (self.nof_bits-1) - 1)

        return x_q.int()

    def dequantizer(self, x_q):
        """
        Dequantizes the input tensor using the calculated scale and zero-point.

        Args:
            x_q (torch.Tensor): The quantized tensor to dequantize.

        Returns:
            torch.Tensor: The dequantized tensor.
        """

        if self.scale is None or self.zero_point is None:
            raise ValueError(f"Scale and zero-point must be calculated before dequantizing [{self.name}].")

        # Dequantize the input
        x_fp = (self.scale) * (x_q.float() - self.zero_point)
        # x_fp = (self.scale) * (x_q.float())
        return x_fp


In [ ]:


# class MinMaxObserver(ObserverBase):
#     def __init__(self,
#                  dtype=torch.qint8,
#                  qscheme=torch.per_tensor_affine,
#                  nof_bits=8,
#                  is_calibrate=False,
#                  name='base'):
#         """
#         MinMaxObserver class is used to record the minimum and maximum values of the input activations.
#         These values will be used to compute the scale and zero-point for quantization.

#         Args:
#             dtype (torch.dtype): The data type for quantization, e.g., torch.qint8.
#             qscheme (torch.qscheme): The quantization scheme, e.g., torch.per_tensor_affine.
#             nof_bits (int): Number of bits for quantization.
#             is_calibrate (bool): Flag indicating whether to calculate the scale and zero-point during forwarding.
#         """
#         super(MinMaxObserver, self).__init__()
#         self.dtype = dtype
#         self.qscheme = qscheme
#         self.nof_bits = nof_bits
#         self.is_calibrate = is_calibrate
#         self.min_val = torch.tensor(float('inf'))  # Initialize min with infinity
#         self.max_val = torch.tensor(float('-inf'))  # Initialize max with -infinity
#         self.zero_point = None
#         self.scale = None
#         self.name = name
#         self.ez = None
#         self.nml = None
#         self.sz  = None

#     def forward(self, x):
#         """
#         This method updates the minimum and maximum values by comparing the current min and max
#         with those in the input tensor. If is_calibrate is True, it calculates the scale and zero-point.

#         Args:
#             x (torch.Tensor): The input tensor to observe.
#         """
#         # global_min = x.min()
#         # global_max = x.max()

#         # if global_min < self.min_val:
#         #     self.min_val = global_min
#         # if global_max > self.max_val:
#         #     self.max_val = global_max
#         gmin = x.amin()   # or torch.min(x)
#         gmax = x.amax()   # or torch.max(x)
#         # no Python conditionals:
#         with torch.no_grad():  # avoid autograd on stats
#             self.min_val = torch.minimum(self.min_val, gmin)
#             self.max_val = torch.maximum(self.max_val, gmax)

#         # Calculate scale and zero-point during forwarding if is_calibrate is True
#         self.calculate_qparams()
#         self.ez = x.element_size()
#         self.nml = x.numel()
#         self.sz = x.size()

#         return x

#     def calculate_qparams(self):
#         """
#         This method computes the scale and zero-point for quantization based on the min and max values
#         observed during the forward pass.

#         Returns:
#             scale (torch.Tensor): The quantization scale.
#             zero_point (torch.Tensor): The quantization zero-point.
#         """
#         rng = self.max_val == self.min_val


#         # if self.min_val == self.max_val:
#         #     # To handle cases where min == max to avoid division by zero
#         #     self.scale = torch.tensor([1.0], dtype=torch.float32)
#         #     self.zero_point = torch.tensor([0], dtype=torch.int32)
#         # else:
#         #     # Calculate scale and zero-point for qint8 quantization
#         #     self.scale = (self.max_val - self.min_val) / (2 ** self.nof_bits - 1)
#         self.scale = torch.where(
#             rng,
#             torch.tensor([1.0], dtype=torch.float32).to("cuda"),
#             (self.max_val - self.min_val) / (2 ** self.nof_bits - 1)
#         )
#         rng_offset = torch.tensor(self.qscheme == torch.per_tensor_affine).to("cuda") or rng
#         # if self.qscheme == torch.per_tensor_affine:
#         self.zero_point = torch.where(
#             rng_offset,
#             torch.tensor([0], dtype=torch.int32).to("cuda"),
#             torch.round(-self.min_val / self.scale).clamp(0, 2 ** self.nof_bits - 1)
#         )
#         # else:
#         #     self.zero_point = 0
#         return self.scale, self.zero_point

#     def quantizer(self, x, scale=None):
#         """
#         Quantizes the input tensor using the calculated scale and zero-point.

#         Args:
#             x (torch.Tensor): The input tensor to quantize.

#         Returns:
#             torch.Tensor: The quantized tensor.
#         """
#         # if self.is_calibrate:
#         # if self.scale is None or self.zero_point is None:
#         #     raise ValueError(f"Scale and zero-point must be calculated before quantizing [{self.name}].")

#         # Quantize the input
#         if self.qscheme == torch.per_tensor_affine:
#             x_q = (x / (self.scale) + self.zero_point).round().clamp(0, 2 ** self.nof_bits - 1)
#         else:
#             x_q = (x / (self.scale) + self.zero_point).round().clamp(-2 ** (self.nof_bits-1), 2 ** (self.nof_bits-1) - 1)

#         return x_q.int()

#     def dequantizer(self, x_q):
#         """
#         Dequantizes the input tensor using the calculated scale and zero-point.

#         Args:
#             x_q (torch.Tensor): The quantized tensor to dequantize.

#         Returns:
#             torch.Tensor: The dequantized tensor.
#         """

#         # if self.scale is None or self.zero_point is None:
#         #     raise ValueError(f"Scale and zero-point must be calculated before dequantizing [{self.name}].")

#         # Dequantize the input
#         x_fp = (self.scale) * (x_q.float() - self.zero_point)
#         # x_fp = (self.scale) * (x_q.float())
#         return x_fp


## Quantization layers

### Auxiliry functions

In [ ]:
def safe_shift_l(x, shift):
    shift_left = torch.clamp(shift, min=0)
    shift_right = torch.clamp(-shift, min=0)
    return (x << shift_left) >> shift_right

def safe_shift_r(x, shift):
    shift_left = torch.clamp(shift, min=0)
    shift_right = torch.clamp(-shift, min=0)
    return (x >> shift_left) << shift_right


In [ ]:
list_dist_exp = []
def build_exp_lut(nof_bits=16, LUT_SIZE=16):
  scale = 1 << (nof_bits - 1)
  exp_lut = torch.zeros(LUT_SIZE + LUT_SIZE, dtype=torch.int64)
  for i, offset in enumerate(range(-LUT_SIZE, LUT_SIZE)):
      exp_lut[i] = torch.round(torch.exp(torch.tensor(offset).float()) * scale).to(dtype=torch.int32)

  return exp_lut


def build_ln_lut(nof_bits=16, LUT_SIZE=16, eps=1e-5):
  scale = 1 << (nof_bits - 1)
  ln_lut = torch.zeros(LUT_SIZE + 1, dtype=torch.int64)
  ln_lut[0] = torch.round(torch.log(torch.tensor(0.25+eps).float()) * scale).to(dtype=torch.int32)  # TODO try to replace with 0.5
  for i, offset in enumerate(range(1, LUT_SIZE)):
      ln_lut[i] = torch.round(torch.log(torch.tensor(2 ** (offset-1) + eps).float()) * scale).to(dtype=torch.int32)

  return ln_lut


def int_approx_lower_power_of_two(x: torch.Tensor):
    # Handle zeros explicitly
    lut_idx = torch.where(
        x == 0,
        torch.tensor(0, dtype=torch.int32, device=x.device),  # return 0 for x==0
        # torch.floor(torch.log2(torch.clamp(x, min=0.25))).to(torch.int32)
        x.log2().floor().to(torch.int32)
    )
    return lut_idx


def TylorNLog(in_x, ev_point=-1, nof_bits=16, iterations=3, LUT_SIZE=16, ln_lut=None, LN2=None):
      """
      Approximate natural logarithm (ln) using a Taylor series expansion around evaluation points.

      Args:
          in_x (torch.Tensor): Input tensor.
          ev_point (int): Evaluation point for the Taylor series expansion. Default is -1 (automatically determined).
          nof_bits (int): Number of bits for quantization.
          iterations (int): Number of iterations for the Taylor series expansion.
          LUT_SIZE (int): Size of the lookup table (LUT).

      Returns:
          torch.Tensor: Approximated logarithmic value using Taylor series.
      """
      dtype = torch.int32
      dtype_w = dtype = torch.int32

      scale = torch.tensor(1 << (nof_bits - 1)).to(dtype)

      # emulated lut
      lut_idx = (int_approx_lower_power_of_two(in_x >> (nof_bits - 4)) - 3).clamp_(-4, LUT_SIZE + 4)

      # k = torch.tensor(lut_idx << (nof_bits - 1)).to(dtype)
      k = (lut_idx << (nof_bits - 1)).to(dtype)
      sum_result = ((k>>1)+(k>>3)+(k>>4))

      if iterations == 0:
          return sum_result

      x_diff = (in_x - (1 << (nof_bits + lut_idx - 1))).to(dtype_w)
      div_scale = (1 << (15 - lut_idx)).to(dtype)
      x_pow = ((x_diff * div_scale) >> (15)).to(dtype)

      sum_result += x_pow

      if iterations == 1:
          return sum_result

      x_pow2 = ((x_pow * x_pow) >> (nof_bits)).to(dtype)
      sum_result -= x_pow2

      if iterations == 2:
          return sum_result

      x_pow3 = ((x_pow2 * x_pow) // (scale * 3)).to(dtype)
      sum_result += x_pow3

      return sum_result

def new_ln(xq, bits):
    aq = xq.log2().floor().int() - bits
    # compute xq >> aq if aq > 0 else xq << -aq)
    k1 = safe_shift_r(xq, aq)
    # compute 2**(-aq))*xq
    k2 = ((aq-1) << bits)
    # compute (2**(-aq))*xq + ((aq-1)*2**bits)
    k = (k1 + k2)
    # yq = ln2q * ((2**(-aq))*xq + ((aq-1)*2**bits))
    yq = (((k>>1)+(k>>3)+(k>>4)))
    return yq

def TaylorExponent(x_scale, scale, iterations=1, input_bits=16, output_bits=16,LUT_SIZE=16, exp_lut=None):
      dtype = torch.int32
      udtype = torch.int32

      # Offset in steps of 2^(input_bits-1) ~ round(x)
      offset = ((x_scale + (1 << (input_bits - 2))) >> (input_bits - 1)).to(dtype)
      offset_clamped = offset.clamp_(-LUT_SIZE, LUT_SIZE)
      # lut_index = (offset_clamped + LUT_SIZE).to(torch.int64)

      # exp_offset = exp_lut[lut_index].to(udtype)
      exp_offset = (offset_clamped.exp() * (1 << (output_bits - 1))).round().to(dtype=dtype)
      if iterations == 0:
          return exp_offset

      x_a = x_scale - (offset << (input_bits - 1))
      x_pow = x_a

      # as provided: 1+x, then optional terms
      sum_result = (x_pow + scale)

      if iterations == 1:
          return (sum_result * exp_offset).to(udtype) >> (input_bits - 1)

      # note: your x^2 uses >> input_bits
      x_pow2 = (x_pow * x_pow) >> (input_bits)
      sum_result = (sum_result + x_pow2)

      if iterations == 2:
          return (sum_result * exp_offset).to(udtype) >> (input_bits - 1)

      x_pow3 = (x_pow2 * x_pow) // ( (1 << (input_bits - 1)) * 3)  # scale = 2^(bits-1)
      sum_result = (sum_result + x_pow3)

      return (sum_result * exp_offset).to(udtype) >> (input_bits - 1)





### QuantizedLinear

In [ ]:
class QuantizedLinear(nn.Linear):
    def __init__(self,
                 in_features,
                 out_features,
                 bias=True,
                 nof_bits1=8,
                 nof_bits2=8,
                 qscheme=torch.per_tensor_affine,
                 is_calibrate=False,
                 is_opt_scale=False,
                 quant=False):
        """
        A quantized version of nn.Linear that uses MinMaxObserver to quantize weights, bias, and inputs.

        Args:
            in_features (int): Number of input features.
            out_features (int): Number of output features.
            bias (bool): If set to False, the layer will not learn an additive bias. Default: True.
            dtype (torch.dtype): Data type for quantization, typically qint8 or quint8.
            qscheme (torch.qscheme): Quantization scheme, typically per_tensor_affine.
            nof_bits (int): Number of bits for quantization.
            is_calibrate (bool): If True, the observer will update min/max values but not quantize the input.
        """
        super(QuantizedLinear, self).__init__(in_features, out_features, bias)

        self.nof_bits1 = nof_bits1
        self.nof_bits2 = nof_bits2
        self.nof_bits_b = nof_bits1 + nof_bits2

        # Initialize observers for inputs, weights, and bias
        self.in_obs = MinMaxObserver(qscheme=torch.per_tensor_affine, nof_bits=nof_bits1, is_calibrate=True, name="linear")
        self.w_obs = MinMaxObserver(qscheme=torch.per_tensor_symmetric, nof_bits=nof_bits2, is_calibrate=True, name="linear")
        self.b_obs = MinMaxObserver(qscheme=torch.per_tensor_symmetric, nof_bits=self.nof_bits_b, is_calibrate=True, name="linear")

        self.is_calibrate = is_calibrate
        self.is_opt_scale = is_opt_scale
        self.is_weights_quantized = False  # Track whether weights have been quantized
        self.quant = quant  # Control whether to perform quantized operations

    def quantize_weights_and_bias(self):
        """
        Quantizes weights and bias if they have not been quantized yet.
        Sets scale and zero-point for weights and bias, and stores quantized values.
        """
        if not self.is_weights_quantized:
            # Quantize the weights
            self.w_obs(self.weight)
            self.scale_weight, self.zero_point_weight = self.w_obs.calculate_qparams()
            self.weight_integer = self.w_obs.quantizer(self.weight) - self.zero_point_weight

            # Quantize the bias if it exists
            if self.bias is not None:
                # Calculate bias quantization scale as the product of input and weight scales
                self.b_obs.scale = self.in_obs.scale * self.w_obs.scale
                self.b_obs.zero_point = 0  # Bias zero-point is typically set to 0
                self.bias_integer = self.b_obs.quantizer(self.bias)
            else:
                # Set scale and zero-point for cases without a bias
                self.b_obs.scale = self.in_obs.scale * self.w_obs.scale
                self.b_obs.zero_point = 0
                self.bias_integer = None

            # Mark weights and bias as quantized
            self.is_weights_quantized = True

    def forward_pass(self, x):
        """
        Forward pass with quantization:
        - Quantizes weights and bias if not already done.
        - Quantizes the input.
        - Performs linear transformation with quantized weights and bias.
        - Dequantizes the output before returning.
        """
        # Ensure weights and bias are quantized
        self.quantize_weights_and_bias()

        # Quantize the input
        self.scale_input, self.zero_point_input = self.in_obs.calculate_qparams()
        _x_q = self.in_obs.quantizer(x)  # Quantize the input
        x_q = _x_q - self.zero_point_input.int()  # Adjust input by zero point

        # Integer linear transformation using quantized weights and bias
        if self.bias is not None:
            output_q = F.linear(
                x_q.float(),
                weight=self.weight_integer.float(),
                bias=self.bias_integer.float()
            )
        else:
            output_q = F.linear(
                x_q.float(),
                weight=self.weight_integer.float()
            )

        # Dequantize the output
        dq_output = self.b_obs.dequantizer(output_q)
        return dq_output

    def float_forward_pass(self, x):
        return F.linear(
            x,
            self.weight,
            self.bias
        )

    def forward(self, x):
        """
        Forward pass through the quantized linear layer.
        If is_calibrate is True, update the min/max values of the input observer without performing quantization.
        Otherwise, quantize the input, weights, and bias, perform integer-based linear operation, and dequantize the result.

        Args:
            x (torch.Tensor): Input tensor to the linear layer.

        Returns:
            torch.Tensor: Dequantized output after quantized linear transformation.
        """
        # Calibration mode: update the input observer and return the input tensor as-is
        if self.is_calibrate:
            self.in_obs(x)
            return self.float_forward_pass(x)

        if self.is_opt_scale:
            self.opt_input = x
            return self.float_forward_pass(x)

        # Apply the linear transformation in the quantized domain using integer arithmetic
        if self.quant:
            return self.forward_pass(x)
        else:
            return self.float_forward_pass(x)

    def set_calibration_flag(self):
        self.is_calibrate = True

    def unset_calibration_flag(self):
        self.is_calibrate = False

    def set_quant(self):
        self.quant = True

    def unset_quant(self):
        self.quant = False

    def set_scale_opt(self):
        self.is_opt_scale = True

    def unset_scale_opt(self):
        del self.opt_input
        self.is_opt_scale = False



### QuantizedMatmul

In [ ]:
class QuantizedMatmul(nn.Module):
    """
    A quantized matrix multiplication layer using integer-only arithmetic for efficiency.
    This class uses MinMaxObserver to quantize inputs and perform matrix multiplication in the quantized domain.
    """

    def __init__(self,
                 is_calibrate=False,
                 in1_bits=8,
                 in2_bits=8,
                 is_opt_scale=False,
                 quant=False):
        """
        Initializes the quantized matrix multiplication layer.

        Args:
            bias (bool): If set to False, the layer will not use bias. Default: True.
            is_calibrate (bool): Flag for enabling calibration mode to update observers.
        """
        super(QuantizedMatmul, self).__init__()

        self.in1_bits=in1_bits
        self.in2_bits=in2_bits
        self.is_calibrate = is_calibrate
        self.quant = quant  # Whether to use quantized operations
        self.is_weights_quantized = False
        self.is_opt_scale = is_opt_scale

        # Initialize observers for two input tensors and the output
        self.in1_obs = MinMaxObserver(qscheme=torch.per_tensor_affine,
                                      nof_bits=self.in1_bits,
                                      is_calibrate=is_calibrate,
                                      name="MatMul")
        self.in2_obs = MinMaxObserver(qscheme=torch.per_tensor_affine,
                                      nof_bits=self.in2_bits,
                                      is_calibrate=is_calibrate,
                                      name="MatMul")
        self.out_obs = MinMaxObserver(qscheme=torch.per_tensor_symmetric,
                                      nof_bits=(self.in1_bits+self.in2_bits),
                                      is_calibrate=is_calibrate,
                                      name="MatMul")

    def quantize_weights_and_bias(self):
        """
        Quantizes weights and bias if they have not been quantized yet.
        Sets scale and zero-point for weights and bias, and stores quantized values.
        """
        if not self.is_weights_quantized:

            # Calculate bias quantization scale as the product of input and weight scales
            self.out_obs.scale = self.in1_obs.scale * self.in2_obs.scale
            self.out_obs.zero_point = 0  # Bias zero-point is typically set to 0

            # Mark weights and bias as quantized
            self.is_weights_quantized = True


    def forward_pass(self, x1, x2):
        self.quantize_weights_and_bias()

        # Quantize the inputs
        scale_input1, zero_point_input1 = self.in1_obs.calculate_qparams()
        scale_input2, zero_point_input2 = self.in2_obs.calculate_qparams()

        # Quantize the inputs based on their scales and zero points
        x_q1 = self.in1_obs.quantizer(x1)
        x_q2 = self.in2_obs.quantizer(x2)

        # Perform quantized matrix multiplication
        output_q = (x_q1 - zero_point_input1) @ (x_q2 - zero_point_input2)

        # Dequantize the output
        dq_output = self.out_obs.dequantizer(output_q)
        return dq_output

    def float_forward_pass(self, x1, x2):
        return x1 @ x2

    def forward(self, x1, x2):
        """
        Forward pass for matrix multiplication.

        Args:
            x1 (torch.Tensor): The first input tensor.
            x2 (torch.Tensor): The second input tensor.

        Returns:
            torch.Tensor: Result of matrix multiplication, either quantized or floating-point.
        """
        # Calibration mode: update observers with input min/max values
        if self.is_calibrate:
            self.in1_obs(x1)
            self.in2_obs(x2)
            return self.float_forward_pass(x1, x2)  # Return the floating-point result during calibration

        if self.is_opt_scale:
            self.opt_input1 = x1
            self.opt_input2 = x2
            return self.float_forward_pass(x1, x2)  # Return the floating-point result during calibration

        # Quantized matrix multiplication
        if self.quant:
            return self.forward_pass(x1, x2)
        else:
            # Standard floating-point matrix multiplication
            return self.float_forward_pass(x1, x2)

    def set_calibration_flag(self):
        """Enable calibration mode to update observers."""
        self.is_calibrate = True

    def unset_calibration_flag(self):
        """Disable calibration mode."""
        self.is_calibrate = False


    def set_quant(self):
        self.quant = True

    def unset_quant(self):
        self.quant = False

    def set_scale_opt(self):
        self.is_opt_scale = True

    def unset_scale_opt(self):
        del self.opt_input1
        del self.opt_input2
        self.is_opt_scale = False


### Softmax

In [ ]:
sf_layer_idx = 0
class IntSoftmaxTS(nn.Module):
    """
    Quantized Softmax with a combination of Lookup Table (LUT) approximation and Taylor Series expansion.
    This class uses integer-only arithmetic to perform the softmax function with quantized inputs.
    """

    def __init__(self,
                 quant=False,
                 is_calibrate=False,
                 nof_bits=16,
                 LUT_SIZE=16,
                 is_opt_scale=False,
                 eps=1e-5,
                 dim=-1,
                 iterations=1,
                 ts_ln=TylorNLog,
                 int_exp_scaled=TaylorExponent):
        super(IntSoftmaxTS, self).__init__()

        self.LUT_SIZE = LUT_SIZE
        self.nof_bits = nof_bits
        self.iterations = iterations
        self.dim = dim
        self.quant = quant
        self.is_calibrate = is_calibrate
        self.is_opt_scale = is_opt_scale
        self.eps = eps

        scale = 1 << (self.nof_bits - 1)
        self.div3 = int(round(scale / 3))

        self.ts_ln = ts_ln
        self.int_exp_scaled = int_exp_scaled

        self.input_bits = self.nof_bits - 4
        self.output_bits = self.nof_bits + 1
        self.stats = dict()
        self.stats[f'softmax'] = []

        self.ln2 = (torch.tensor(2).log() * 2 ** (self.output_bits-1)).round().to(torch.int32)

        self.in_obs = MinMaxObserver(qscheme=torch.per_tensor_symmetric,
                                     nof_bits=self.nof_bits,
                                     is_calibrate=is_calibrate,
                                     name="Softmax")

        self.exp_lut = []
        self.ln_lut =  []

        # self.exp_lut = build_exp_lut(nof_bits=self.output_bits, LUT_SIZE=self.LUT_SIZE)
        # self.ln_lut = build_ln_lut(nof_bits=self.output_bits, LUT_SIZE=self.LUT_SIZE, eps=self.eps)

        # if torch.cuda.is_available():
        #     self.exp_lut = self.exp_lut.cuda()
        #     self.ln_lut = self.ln_lut.cuda()



    def int_softmax(self, x):

        dtype = torch.int32
        udtype = torch.int32

        self.scale_input, self.zero_point_input = self.in_obs.calculate_qparams()
        _x_q = self.in_obs.quantizer(x)  # Quantize the input
        x_q = _x_q  # Adjust input by zero point
        # print(x[0][0][0])
        # print(x_q[0][0][0])
        # print(self.in_obs.dequantizer(x_q))
        # print("--------------------------")
        # reference
        # x_int = x - x.max(dim=self.dim, keepdim=True).values

        # rounding float
        # x_round = x.round()
        # x_int = x_round - x_round.max(dim=self.dim, keepdim=True).values

        # quantized 4 bits
        x_int = self.in_obs.dequantizer(x_q - x_q.max(dim=self.dim, keepdim=True).values).round()

        spacial_scale = 1 << (self.input_bits - 1)
        x_scale = (x_int * spacial_scale).floor().to(dtype)

        try:
            exp_int = self.int_exp_scaled(x_scale,
                                          spacial_scale,
                                          input_bits=self.input_bits,
                                          output_bits=self.output_bits,
                                          LUT_SIZE=self.LUT_SIZE,
                                          exp_lut=self.exp_lut,
                                          iterations=0)  # TODO test zero iterations
                                          # iterations=self.iterations)
            # if self.is_float_exp == True:
            #   exp_int = ((x - x.max(dim=self.dim, keepdim=True).values).exp() * 2**(self.output_bits - 1)).floor().to(dtype)

            # spacial_scale_out = 1 << (self.output_bits - 1)
            # x_scale = (x_int * spacial_scale_out).floor().to(dtype)
            # y = x_scale + (x_scale>>1) - (x_scale>>4)
            # z1 = y//spacial_scale_out
            # exp_int = torch.where(z1>0, spacial_scale_out<<z1, spacial_scale_out>>-z1)

        except RuntimeError as err:
            print("[EXP 1] : TEST")
            print(x_int)
            raise err
        exp_int_sum = exp_int.sum(dim=-1, keepdim=True)

        # ln_sum = self.ts_ln(exp_int_sum,
        #                     # iterations=self.iterations+1,
        #                     iterations=2,
        #                     nof_bits=self.output_bits,
        #                     ln_lut=self.ln_lut,
        #                     LN2=self.ln2)
        ln_sum = new_ln(exp_int_sum, self.output_bits-1)

        spacial_scale = 1 << (self.output_bits - 1)

        try:
            ln_mul = self.int_exp_scaled(-ln_sum.int(),
                                        spacial_scale,
                                        input_bits=self.output_bits,
                                        output_bits=self.output_bits,
                                        LUT_SIZE=self.LUT_SIZE,
                                        exp_lut=self.exp_lut,
                                        iterations=1)
                                        # iterations=self.iterations)
        except RuntimeError as err:
            print("[EXP 2] : TEST")
            print(x_int)
            raise err
        # TODO: add normal divider case
        # ln_mul = (1 << ((self.output_bits - 1) + (self.output_bits - 1)) )// exp_int_sum

        sf_values = ln_mul * exp_int
        return sf_values

    def foward_pass(self, x):
        sf_values = self.int_softmax(x)
        deq_softmax = sf_values / (1 << ((self.output_bits - 1) + (self.output_bits - 1)))
        return deq_softmax

    def float_forward_pass(self, x):
        return x.softmax(dim=-1)


    def forward(self, x):
        if self.is_calibrate:
            self.in_obs(x);  return x.softmax(dim=self.dim)

        if self.is_opt_scale:
            self.opt_input = x;  return x.softmax(dim=self.dim)

        # self.stats[f'softmax'].append(x)

        if self.quant:
            return self.foward_pass(x)
        else:
            return self.float_forward_pass(x)

    def set_calibration_flag(self):
        self.is_calibrate = True

    def unset_calibration_flag(self):
        self.is_calibrate = False

    def set_quant(self):
        self.quant = True

    def unset_quant(self):
        self.quant = False

    def set_scale_opt(self):
        self.is_opt_scale = True

    def unset_scale_opt(self):
        del self.opt_input
        self.is_opt_scale = False


### qHadamardProd

In [ ]:
class qHadamardProd(nn.Module):
    """
    A quantized matrix multiplication layer using integer-only arithmetic for efficiency.
    This class uses MinMaxObserver to quantize inputs and perform matrix multiplication in the quantized domain.
    """

    def __init__(self,
                 nof_bits1=16,
                 nof_bits2=16,
                 quant=False,
                 is_calibrate=False):
        """
        Initializes the quantized matrix multiplication layer.

        Args:
            bias (bool): If set to False, the layer will not use bias. Default: True.
            is_calibrate (bool): Flag for enabling calibration mode to update observers.
        """
        super(qHadamardProd, self).__init__()
        self.nof_bits1=nof_bits1
        self.nof_bits2=nof_bits2
        self.nof_bits_o = self.nof_bits1 + self.nof_bits2

        # Initialize observers for two input tensors and the output
        self.in1_obs = MinMaxObserver(qscheme=torch.per_tensor_affine,
                                      nof_bits=self.nof_bits1,
                                      is_calibrate=is_calibrate,
                                      name="hadamard")
        self.in2_obs = MinMaxObserver(qscheme=torch.per_tensor_affine,
                                      nof_bits=self.nof_bits2,
                                      is_calibrate=is_calibrate,
                                      name="hadamard")
        self.out_obs = MinMaxObserver(qscheme=torch.per_tensor_symmetric,
                                      nof_bits=self.nof_bits_o,
                                      is_calibrate=is_calibrate,
                                      name="hadamard")

        self.is_calibrate = is_calibrate
        self.quant = quant  # Whether to use quantized operations
        self.is_out_scaled = False

    def set_out_scale(self):
        """
        Quantizes weights and bias if they have not been quantized yet.
        Sets scale and zero-point for weights and bias, and stores quantized values.
        """
        if not self.is_out_scaled:

            # Calculate bias quantization scale as the product of input and weight scales
            self.out_obs.scale = self.in1_obs.scale * self.in2_obs.scale
            self.out_obs.zero_point = 0  # Bias zero-point is typically set to 0

            # Mark weights and bias as quantized
            self.is_out_scaled = True

    def float_forward_pass(self, x1, x2):
        return x1 * x2

    def forward_pass(self, x1, x2):
        self.set_out_scale()

        # Quantize the inputs
        _, zero_point_input1 = self.in1_obs.calculate_qparams()
        _, zero_point_input2 = self.in2_obs.calculate_qparams()

        # Quantize the inputs based on their scales and zero points
        x_q1 = self.in1_obs.quantizer(x1)
        x_q2 = self.in2_obs.quantizer(x2)

        # Perform quantized matrix multiplication
        output_q = (x_q1 - zero_point_input1) * (x_q2 - zero_point_input2)

        # Dequantize the output
        dq_output = self.out_obs.dequantizer(output_q)
        return dq_output

    def forward(self, x1, x2):
        """
        Forward pass for matrix multiplication.

        Args:
            x1 (torch.Tensor): The first input tensor.
            x2 (torch.Tensor): The second input tensor.

        Returns:
            torch.Tensor: Result of matrix multiplication, either quantized or floating-point.
        """
        # Calibration mode: update observers with input min/max values
        if self.is_calibrate:
            self.in1_obs(x1)
            self.in2_obs(x2)
            return self.float_forward_pass(x1, x2)  # Return the floating-point result during calibration

        # Quantized matrix multiplication
        if self.quant:
            return self.forward_pass(x1, x2)
        else:
            # Standard floating-point matrix multiplication
            return self.float_forward_pass(x1, x2)


    def set_calibration_flag(self):
        """Enable calibration mode to update observers."""
        self.is_calibrate = True

    def unset_calibration_flag(self):
        """Disable calibration mode."""
        self.is_calibrate = False

    def set_quant(self):
        self.quant = True

    def unset_quant(self):
        self.quant = False

    def set_scale_opt(self):
        self.is_opt_scale = True

    def unset_scale_opt(self):
        # del self.opt_input
        self.is_opt_scale = False


### IntGeluTS

In [ ]:
class IntGeluTS(nn.Module):
    """
    Quantized GELU using a combination of Lookup Table (LUT) approximation and Taylor Series expansion.
    This class uses integer-only arithmetic to perform the GELU function with quantized inputs.
    """

    def __init__(self,
                 quant=False,
                 LUT_SIZE=16,
                 nof_bits=16,
                 eps=1e-5,
                 gelu_scale=1.702,
                 is_opt_scale=False,
                 is_calibrate=False,
                 ts_ln=TylorNLog,
                 int_exp_scaled=TaylorExponent):
        super(IntGeluTS, self).__init__()

        self.LUT_SIZE = LUT_SIZE
        self.nof_bits = nof_bits
        self.iterations = 2
        self.is_opt_scale = is_opt_scale
        self.is_weights_quantized = True
        self.gelu_scale = gelu_scale
        self.ts_ln = ts_ln
        self.int_exp_scaled = int_exp_scaled
        self.eps = eps

        self.input_bits = self.nof_bits - 4
        self.output_bits = self.nof_bits + 1

        # self.ln2 = (torch.tensor(2).log() * 2 ** (self.output_bits-1)).round().to(torch.int32)

        self.quant = quant
        self.is_calibrate = is_calibrate

        self.had_mul = qHadamardProd(nof_bits1=nof_bits,
                                     nof_bits2=nof_bits,
                                     quant=quant)

        self.in_obs = MinMaxObserver(qscheme=torch.per_tensor_affine,
                                     nof_bits=nof_bits,
                                     is_calibrate=True,
                                     name="geLU")

        self.k_values = torch.tensor([
            2.834596, 2.338217, 1.978175, 1.754823, 1.642511,
            1.642511, 1.754823, 1.978175, 2.338217, 2.834596
        ]).cuda()

        self.exp_lut = []
        self.ln_lut =  []

        # self.exp_lut = build_exp_lut(nof_bits=self.output_bits, LUT_SIZE=self.LUT_SIZE)
        # self.ln_lut = build_ln_lut(nof_bits=self.output_bits, LUT_SIZE=self.LUT_SIZE, eps=self.eps)

        # if torch.cuda.is_available():
        #     self.exp_lut = self.exp_lut.cuda()
        #     self.ln_lut = self.ln_lut.cuda()
        #     self.k_values = self.k_values.cuda()

    def int_sigmoid(self, x, nof_bits=16, iterations=2, gelu_scale=1.702, LUT_SIZE=-1):
        input_bits = nof_bits - 4
        output_bits = nof_bits

        x_sig = x * gelu_scale
        x_max = torch.clamp(x_sig, min=0)
        x_int = x_sig - x_max

        spacial_scale = 1 << (input_bits - 1)
        x_scale = (x_int * spacial_scale).floor().to(dtype=torch.int32)
        x_max_scale = -(x_max * spacial_scale).floor().to(dtype=torch.int32)

        exp_int = TaylorExponent(x_scale,
                              spacial_scale,
                              input_bits=input_bits,
                              output_bits=output_bits,
                              LUT_SIZE=LUT_SIZE,
                              exp_lut=[],
                              iterations=0)
                              # iterations=iterations)

        exp_zero = TaylorExponent(x_max_scale,
                              spacial_scale,
                              input_bits=input_bits,
                              output_bits=output_bits,
                              LUT_SIZE=LUT_SIZE,
                              exp_lut=[],
                              iterations=0)

        exp_int_sum = exp_int + exp_zero

        return exp_int, exp_int_sum

    def get_k_value(self, x):
        x = torch.clamp(x, -5, 5)
        x_shifted = x + 5
        indices = torch.clamp(x_shifted.floor().int(), 0, 9)
        return self.k_values[indices]

    def forward_pass(self, x):
        alpha = self.get_k_value(x)
        # alpha = 1.702  # self.get_k_value(x)

        exp_int, exp_int_sum = self.int_sigmoid(x, self.nof_bits, self.iterations, alpha, self.LUT_SIZE)

        # ln_sum = self.ts_ln(exp_int_sum,
        #                     iterations=self.iterations+1,
        #                     nof_bits=self.output_bits,
        #                     ln_lut=self.ln_lut,
        #                     LN2=None)

        ln_sum = new_ln(exp_int_sum, self.output_bits-1)

        spacial_scale_out = 1 << (self.output_bits - 1)

        ln_mul = self.int_exp_scaled(-ln_sum,
                                     spacial_scale_out,
                                     input_bits=self.output_bits,
                                    output_bits=self.output_bits,
                                    LUT_SIZE=self.LUT_SIZE,
                                    exp_lut=[],
                                    iterations=1)

        q_sigmoid = ln_mul * exp_int
        deq_sigmoid = q_sigmoid / (1 << ((self.output_bits - 1) + (self.output_bits - 1)))

        return self.had_mul(x, deq_sigmoid)

    def float_forward_pass(self, x):
        normal = torch.distributions.Normal(0.0, 1.0)
        sig = normal.cdf(x)
        return self.had_mul(x, sig)

    def quantize_weights_and_bias(self):
        if not self.is_weights_quantized:
            self.is_weights_quantized = True

    def forward(self, x):
        if self.is_calibrate:
            self.in_obs(x)
            return self.float_forward_pass(x)

        if self.is_opt_scale:
            self.opt_input = x
            return self.float_forward_pass(x)

        return self.forward_pass(x) if self.quant else self.float_forward_pass(x)

    def set_calibration_flag(self):
        self.is_calibrate = True

    def unset_calibration_flag(self):
        self.is_calibrate = False

    def set_quant(self):
        self.quant = True

    def unset_quant(self):
        self.quant = False

    def set_scale_opt(self):
        self.is_opt_scale = True

    def unset_scale_opt(self):
        del self.opt_input
        self.is_opt_scale = False


### lnorm

In [ ]:
class QLayerNorm(nn.LayerNorm):
    """
    A subclass of PyTorch's LayerNorm with quantization support.

    Args:
        normalized_shape: Input shape from an expected input of size (N, *), where * means any number of additional dimensions.
        eps: A value added to the denominator for numerical stability.
        elementwise_affine: When set to True, this module has learnable per-element affine parameters.
    """
    def __init__(self,
                 normalized_shape,
                 eps=1e-5,
                 in1_bits=16,
                 in2_bits=16,
                 LUT_SIZE=16,
                 quant=False,
                 elementwise_affine=True,
                 is_opt_scale=False,
                 is_calibrate=False,
                 ts_ln=TylorNLog,
                 int_exp_scaled=TaylorExponent,
                 iterations=3):

        super(QLayerNorm, self).__init__(normalized_shape, eps, elementwise_affine)
        self.ts_ln = ts_ln
        self.int_exp_scaled = int_exp_scaled

        self.in1_bits = in1_bits
        self.in2_bits = in2_bits-1
        self.is_opt_scale = is_opt_scale
        self.is_calibrate = is_calibrate
        self.LUT_SIZE = LUT_SIZE
        self.quant = quant  # Control whether to perform quantized operations
        self.is_weights_quantized = False  # Track whether weights have been quantized
        self.iterations = iterations
        self.input_bits1 = self.in1_bits
        self.output_bits1 = self.in1_bits

        self.in_obs_normalize = MinMaxObserver(qscheme=torch.per_tensor_affine,
                                     nof_bits=self.input_bits1,
                                     is_calibrate=True,
                                     name="lnorm")

        # Initialize observers for inputs, weights, and self.bias
        self.in_obs = MinMaxObserver(qscheme=torch.per_tensor_affine,
                                     nof_bits=self.input_bits1,
                                     is_calibrate=True,
                                     name="lnorm")
        self.w_obs = MinMaxObserver(qscheme=torch.per_tensor_affine,
                                    nof_bits=self.in2_bits,
                                    is_calibrate=True,
                                    name="lnorm")

        if self.bias is not None:
            self.b_obs = MinMaxObserver(qscheme=torch.per_tensor_symmetric,
                                      nof_bits=self.in1_bits+self.in2_bits,
                                      is_calibrate=True,
                                      name="lnorm")
        else:
            self.b_obs = None

        self.output_bits1 = self.in1_bits
        self.exp_lut = build_exp_lut(nof_bits=self.output_bits1, LUT_SIZE=self.LUT_SIZE)
        self.ln_lut = build_ln_lut(nof_bits=self.output_bits1, LUT_SIZE=self.LUT_SIZE, eps=self.eps)
        self.ln2 = (torch.tensor(2).log() * 2 ** (self.output_bits1-1)).round().to(torch.int32)
        self.stats = dict()

        self.stats['scale'] = []
        self.stats['ref'] = []
        self.stats['var'] = torch.tensor([]).cuda()
        # self.sample_counter = 0
        self.sample_counter = 32 # do not sample

        if torch.cuda.is_available():
            self.exp_lut = self.exp_lut.cuda()
            self.ln_lut = self.ln_lut.cuda()

    # Define ts_ln function
    def scaled_ln16(self, in_x, nof_bits=8):
        scale_factor = 1 << (self.input_bits1 - 1)
        scaled_input = (in_x * scale_factor).floor().to(dtype=torch.int32)

        # log_factor = torch.where(
        #     scaled_input >> (self.output_bits1 - 1) == 0,
        #     2,
        #     0
        # )


        # ln_sum = self.ts_ln(scaled_input << (log_factor << 1),
        #                     iterations=3,
        #                     nof_bits=self.output_bits1,
        #                     ln_lut=self.ln_lut,
        #                     LN2=self.ln2) >> 1


        # out = self.int_exp_scaled(-ln_sum,
        #                           scale_factor,
        #                           input_bits=self.output_bits1,
        #                           output_bits=self.output_bits1,
        #                           LUT_SIZE=self.LUT_SIZE,
        #                           exp_lut=self.exp_lut,
        #                           iterations=2) << log_factor

        ln_sum = new_ln(scaled_input, self.output_bits1-1)

        out = TaylorExponent(-ln_sum>>1,
                             scale_factor,
                             iterations=2,
                             input_bits=self.output_bits1,
                             output_bits=self.output_bits1)

        return out

    def float_foward_pass_normalize(self, x):
        x_mean = x.float().mean(dim=-1, keepdim=True)
        x_var = x.float().var(dim=-1, unbiased=False, keepdim=True)
        x_div = x_var.sqrt()
        normed_x = (x - x_mean) / x_div
        return normed_x

    def quantize_weights_and_bias(self):
        """
        Quantizes weights and bias if they have not been quantized yet.
        Sets scale and zero-point for weights and bias, and stores quantized values.
        """
        if not self.is_weights_quantized:
            # Quantize the weights
            self.w_obs(self.weight)
            self.scale_weight, self.zero_point_weight = self.w_obs.calculate_qparams()
            self.weight_integer = self.w_obs.quantizer(self.weight)
            # print(f"[DEBUG] : self.zero_point_weight: {self.zero_point_weight}, min val: {self.w_obs.min_val}, max val: {self.w_obs.max_val}, scale: {self.scale_weight}")
            # Quantize the bias if it exists
            if self.bias is not None:
                # Calculate bias quantization scale as the product of input and weight scales
                self.b_obs.scale = self.in_obs.scale * self.w_obs.scale
                self.b_obs.zero_point = 0  # bias zero-point is typically set to 0
                self.bias_integer = self.b_obs.quantizer(self.bias)
            else:
                # Set scale and zero-point for cases without a bias
                self.b_obs.scale = self.in_obs.scale * self.w_obs.scale
                self.b_obs.zero_point = 0
                self.bias_integer = None

            # Mark weights and bias as quantized
            self.is_weights_quantized = True

    def betta_gamma_forward_pass(self, x):
        """
        Forward pass with quantization:
        - Quantizes weights and bias if not already done.
        - Quantizes the input.
        - Performs linear transformation with quantized weights and bias.
        - Dequantizes the output before returning.
        """

        # Ensure weights and bias are quantized
        self.quantize_weights_and_bias()

        # Quantize the input
        _ , self.zero_point_input = self.in_obs.calculate_qparams()
        # _ , self.zero_point_weight = self.w_obs.calculate_qparams()
        x_q = self.in_obs.quantizer(x)  # Quantize the input
        x_q = x_q - self.zero_point_input.int()  # Adjust input by zero point

        # Integer linear transformation using quantized weights and bias
        if self.bias is not None:
            output_q = x_q * (self.weight_integer - self.zero_point_weight.int()) + self.bias_integer
        else:
            output_q = x_q * (self.weight_integer - self.zero_point_weight.int())

        # print(self.w_obs.dequantizer(self.weight_integer + self.zero_point_weight.int()))
        # print("-----------------")
        # print(self.weight)
        # print("=================")

        # Dequantize the output
        dq_output = self.b_obs.dequantizer(output_q)
        return dq_output

    def normalize(self, x):
        self.scale_input_normed , self.zero_point_input_normed = self.in_obs_normalize.calculate_qparams()

        x_q = self.in_obs_normalize.quantizer(x)
        x_q = x_q - self.zero_point_input_normed

        xq_mean = x_q.float().mean(dim=-1, keepdim=True).long()
        xq_var = x_q.float().var(dim=-1, unbiased=False, keepdim=True)
        x_var = ((self.scale_input_normed) ** 2) * (xq_var.float())

        ln_mul = self.scaled_ln16(x_var, nof_bits=self.in1_bits)

        q_mean_val = (x_q - xq_mean)
        x_dq = ln_mul * q_mean_val * ((self.scale_input_normed) / 2**(self.in1_bits-1))

        # x_mean = x.float().mean(dim=-1, keepdim=True)
        # x_var = x.float().var(dim=-1, unbiased=False, keepdim=True)
        # x_div = x_var.sqrt()

        return x_dq

    def float_betta_gamma_forward_pass(self, x):
        return x * self.weight + self.bias

    def forward_pass(self, x):
        mean_val = self.normalize(x)
        # mean_val = self.float_foward_pass_normalize(x)
        dq_output = self.betta_gamma_forward_pass(mean_val)
        # dq_output = self.float_betta_gamma_forward_pass(mean_val)
        # self.stats['var'].append(dq_output)
        # self.stats['ref'].append(self.float_betta_gamma_forward_pass(mean_val))

        return dq_output

    def float_forward_pass(self, x):
        if self.sample_counter < 32:
            self.stats['var'] = torch.cat((self.stats['var'], x.var(dim=-1, unbiased=False, keepdim=True)), dim=-1)
            self.sample_counter += 1

        return F.layer_norm(x,
                            self.normalized_shape,
                            weight=self.weight,
                            bias=self.bias,
                            eps=self.eps
        )

    def forward(self, x):

        # Calibration mode: update the input observer and return the input tensor as-is
        if self.is_calibrate:
            self.in_obs_normalize(x)
            normed_x = self.float_foward_pass_normalize(x)
            _ = self.in_obs(normed_x)
            return self.float_forward_pass(x)

        if self.is_opt_scale:
            self.opt_input = x
            return self.float_forward_pass(x)

        if self.quant:
            dq_output = self.forward_pass(x)
            return dq_output
        else:
            return self.float_forward_pass(x)

    def set_calibration_flag(self):
        """Enable calibration mode to update observers."""
        self.is_calibrate = True

    def unset_calibration_flag(self):
        """Disable calibration mode."""
        self.is_calibrate = False

    def set_quant(self):
        self.quant = True

    def unset_quant(self):
        self.quant = False

    def set_scale_opt(self):
        self.is_opt_scale = True

    def unset_scale_opt(self):
        del self.opt_input
        self.is_opt_scale = False


## base model parameters

In [ ]:
# Quantization parameters class
class QauntParams(nn.Module):
    def __init__(self):
        super().__init__()
        self.quant = False
        self.nof_bits_linear1 = 8
        self.nof_bits_linear2 = 8
        self.nof_bits_gelu    = 8
        self.nof_bits_softmax = 8
        self.lut_size_softmax = 7
        self.nof_bits_lnorm1 = 12
        self.nof_bits_lnorm2 = 4
        self.nof_bits_matmul1 = 8
        self.nof_bits_matmul2 = 8


    def set_calibration_flag(self):
        for m in self.modules():
            if type(m) in self.q_module_list:
                m.set_calibration_flag()

    def unset_calibration_flag(self):
        for m in self.modules():
            if type(m) in self.q_module_list:
                m.unset_calibration_flag()

    def set_quant(self):
        for m in self.modules():
            if type(m) in self.q_module_list:
                m.set_quant()

    def unset_quant(self):
        for m in self.modules():
            if type(m) in self.q_module_list:
                m.unset_quant()

    def set_scale_opt(self):
        for m in self.modules():
            if type(m) in self.q_module_list:
            # if type(m) in [QuantizedLinear]:
                m.set_scale_opt()

    def unset_scale_opt(self):
        for m in self.modules():
            if type(m) in self.q_module_list:
            # if type(m) in [QuantizedLinear]:
                m.unset_scale_opt()


# vision transformers

## mlp

In [ ]:
class Mlp(nn.Module):
    """ Feed-forward network (MLP) block used in Transformer blocks """
    def __init__(self, in_features,
                 hidden_features=None,
                 out_features=None,
                 act_layer=IntGeluTS, # just for the protocol
                 is_calibrate=False,
                 quant=False,
                 drop=0.):
        super().__init__()
        self.quant=quant
        out_features = out_features or in_features
        hidden_features = hidden_features or in_features
        self.fc1 = QuantizedLinear(in_features,
                                   hidden_features,
                                   nof_bits1=8,
                                   nof_bits2=8,
                                   quant=quant)

        # self.fc1 = nn.Linear(in_features, hidden_features)
        self.act = act_layer(quant=quant,
                             LUT_SIZE=16,
                             nof_bits=8)

        self.fc2 = QuantizedLinear(hidden_features,
                                   out_features,
                                   nof_bits1=8,
                                   nof_bits2=8,
                                   quant=quant)

        # self.fc2 = nn.Linear(hidden_features, out_features)
        self.drop = nn.Dropout(drop)

    def forward(self, x):
        x = self.fc1(x)
        x = self.act(x)
        x = self.drop(x)
        x = self.fc2(x)
        x = self.drop(x)
        return x

    def set_quant(self):
        self.quant = True

    def unset_quant(self):
        self.quant = False


## attention

In [ ]:
class Attention(QauntParams):
    """ Multi-head self-attention mechanism """
    def __init__(self,
                 dim,
                 num_heads=8,
                 qkv_bias=False,
                 attn_drop=0.,
                 proj_drop=0.,
                 quant=False):
        super().__init__()
        self.num_heads = num_heads
        head_dim = dim // num_heads
        self.scale = head_dim ** -0.5

        self.qkv = QuantizedLinear(dim,
                                   dim * 3,
                                   bias=qkv_bias,
                                   nof_bits1=8,
                                   nof_bits2=8,
                                   quant=quant)
        self.mat_mul_qk = QuantizedMatmul(in1_bits=8,
                                          in2_bits=8,
                                          quant=quant)
        self.sf = IntSoftmaxTS(nof_bits=self.nof_bits_softmax,
                               LUT_SIZE=self.lut_size_softmax,
                               dim=-1,
                               quant=self.quant)

        # self.qkv = nn.Linear(dim, dim * 3, bias=qkv_bias)
        self.attn_drop = nn.Dropout(attn_drop)
        self.mat_mul_pv = QuantizedMatmul(in1_bits=8,
                                          in2_bits=8,
                                          quant=quant)
        self.proj = QuantizedLinear(dim,
                                    dim,
                                    nof_bits1=8,
                                    nof_bits2=8,
                                    quant=quant)
        # self.proj = nn.Linear(dim, dim)
        self.proj_drop = nn.Dropout(proj_drop)

    def forward(self, x):
        B, N, C = x.shape
        qkv = self.qkv(x).reshape(B, N, 3, self.num_heads, C // self.num_heads).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]  # Split into query, key, and value tensors
        attn = self.mat_mul_qk(q, k.transpose(-2, -1)) * self.scale  # Scaled dot-product attention
        # attn = (q @ k.transpose(-2, -1)) * self.scale  # Scaled dot-product attention
        attn = self.sf(attn)
        attn = self.attn_drop(attn)

        x = self.mat_mul_pv(attn, v).transpose(1, 2).reshape(B, N, C)
        # x = (attn @ v).transpose(1, 2).reshape(B, N, C)
        x = self.proj(x)
        x = self.proj_drop(x)
        return x

    def set_quant(self):
        self.quant = True

    def unset_quant(self):
        self.quant = False


## block

In [ ]:
class Block(QauntParams):
    """ Transformer Block: Multi-head Attention + MLP + LayerNorm """
    def __init__(self,
                 embed_dim,
                 num_heads,
                 mlp_ratio=4.,
                 qkv_bias=False,
                 drop=0.,
                 attn_drop=0.,
                 drop_path=0.,
                 is_calibrate = False,
                 act_layer=IntGeluTS,
                 norm_layer=QLayerNorm,
                 quant=False):

        super().__init__()
        self.norm1 = norm_layer(embed_dim,
                                in1_bits=self.nof_bits_lnorm1,
                                in2_bits=self.nof_bits_lnorm2,
                                quant=quant)
        self.attn = Attention(embed_dim,
                              num_heads=num_heads,
                              qkv_bias=qkv_bias,
                              attn_drop=attn_drop,
                              proj_drop=drop,
                              quant=quant)

        # Drop path layer for stochastic depth (not always needed)
        self.drop_path = DropPath(drop_path) if drop_path > 0. else nn.Identity()
        self.norm2 = norm_layer(embed_dim,
                                in1_bits=self.nof_bits_lnorm1,
                                in2_bits=self.nof_bits_lnorm2,
                                quant=quant)

        mlp_hidden_dim = int(embed_dim * mlp_ratio)
        self.mlp = Mlp(in_features=embed_dim,
                       hidden_features=mlp_hidden_dim,
                       act_layer=act_layer,
                       drop=drop,
                       quant=quant)


    def forward(self, x):
        x = x + self.drop_path(self.attn(self.norm1(x)))  # Residual connection after attention
        x = x + self.drop_path(self.mlp(self.norm2(x)))   # Residual connection after MLP
        return x

    def set_quant(self):
        self.quant = True

    def unset_quant(self):
        self.quant = False


## Model features

In [ ]:
# Helper class for DropPath (Stochastic Depth)
class DropPath(nn.Module):
    """ Drop paths (stochastic depth) per sample (when applied in main path of residual blocks). """
    def __init__(self, drop_prob=None):
        super(DropPath, self).__init__()
        self.drop_prob = drop_prob

    def forward(self, x):
        return drop_path(x, self.drop_prob)

def drop_path(x, drop_prob: float = 0., training: bool = False):
    """ Drop paths (Stochastic Depth) per sample (batch-wise) """
    if drop_prob == 0. or not training:
        return x
    keep_prob = 1 - drop_prob
    shape = (x.shape[0],) + (1,) * (x.ndim - 1)
    random_tensor = keep_prob + torch.rand(shape, dtype=x.dtype, device=x.device)
    random_tensor.floor_()  # binarize
    output = x.div(keep_prob) * random_tensor
    return output


class PatchEmbed(nn.Module):
    """ Image to Patch Embedding

    Args:
        img_size (int): Input image size.
        patch_size (int): Size of the patch to split the image into.
        in_chans (int): Number of input image channels (e.g., 3 for RGB).
        embed_dim (int): Dimensionality of the embedding space.
    """
    def __init__(self, img_size=224, patch_size=16, in_chans=3, embed_dim=768):
        super().__init__()
        self.img_size = img_size
        self.patch_size = patch_size
        self.grid_size = img_size // patch_size
        self.num_patches = self.grid_size ** 2  # Number of patches in the image

        # A convolution layer to split the image into patches and project them into the embedding space
        self.proj = nn.Conv2d(in_chans, embed_dim, kernel_size=patch_size, stride=patch_size)

    def forward(self, x):
        """
        Args:
            x (Tensor): Shape (batch_size, in_chans, img_size, img_size)
        Returns:
            Patch embeddings: Shape (batch_size, num_patches, embed_dim)
        """
        # Shape after conv: (batch_size, embed_dim, grid_size, grid_size)
        x = self.proj(x)

        # Flatten the patches and change the shape to (batch_size, num_patches, embed_dim)
        x = x.flatten(2)  # Shape (batch_size, embed_dim, num_patches)
        x = x.transpose(1, 2)  # Shape (batch_size, num_patches, embed_dim)

        return x


In [ ]:
class QuantTransformer(QauntParams):


    def __init__(self,
                 quant=False,
                 is_calibrate = False,
                 q_module_list=[QuantizedLinear, QuantizedMatmul, qHadamardProd, QLayerNorm]):
        super().__init__()
        self.q_module_list = q_module_list
        self.quant = quant
        self.is_calibrate = is_calibrate
        self.q_module_list = q_module_list

    def set_calibration_flag(self):
        for m in self.modules():
            if type(m) in self.q_module_list:
                print(f"[SET CALIBRATION FLAG] type:{type(m)}, module list: {self.q_module_list}")
                m.set_calibration_flag()

    def unset_calibration_flag(self):
        for m in self.modules():
            if type(m) in self.q_module_list:
                print(f"[UNSET CALIBRATION FLAG] type:{type(m)}, module list: {self.q_module_list}")
                m.unset_calibration_flag()

    def set_quant(self):
        for m in self.modules():
            if type(m) in self.q_module_list:
                m.set_quant()

    def unset_quant(self):
        for m in self.modules():
            if type(m) in self.q_module_list:
                m.unset_quant()

    def set_scale_opt(self):
        for m in self.modules():
            if type(m) in self.q_module_list:
            # if type(m) in [QuantizedLinear]:
                m.set_scale_opt()

    def unset_scale_opt(self):
        for m in self.modules():
            if type(m) in self.q_module_list:
            # if type(m) in [QuantizedLinear]:
                m.unset_scale_opt()

## Deit model

In [ ]:
class DistilledVisionTransformer(QuantTransformer):
    def __init__(self, img_size=224,
                 patch_size=16,
                 in_chans=3,
                 num_classes=1000,
                 embed_dim=768,
                 depth=12,
                 num_heads=12,
                 mlp_ratio=4.,
                 qkv_bias=True,
                 distilled=True,
                 quant=False,
                 is_calibrate = False,
                 q_module_list=[QuantizedLinear, QuantizedMatmul, qHadamardProd, QLayerNorm],
                 **kwargs):
        super().__init__(
            quant=quant,
            is_calibrate=is_calibrate,
            q_module_list=q_module_list
        )
        self.num_classes = num_classes
        self.num_features = self.embed_dim = embed_dim  # num_features for consistency with other models
        self.num_tokens = 2 if distilled else 1

        self.patch_embed = PatchEmbed(img_size=img_size,
                                      patch_size=patch_size,
                                      in_chans=in_chans,
                                      embed_dim=embed_dim)
        num_patches = self.patch_embed.num_patches

        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        self.dist_token = nn.Parameter(torch.zeros(1, 1, embed_dim)) if distilled else None
        self.pos_embed = nn.Parameter(torch.zeros(1, num_patches + self.num_tokens, embed_dim))
        self.pos_drop = nn.Dropout(p=0.1)

        self.blocks = nn.ModuleList([
            Block(embed_dim=embed_dim,
                  num_heads=num_heads,
                  mlp_ratio=mlp_ratio,
                  qkv_bias=qkv_bias,
                  is_calibrate=self.is_calibrate,
                  act_layer=IntGeluTS,
                  norm_layer=QLayerNorm,
                  quant=self.quant,
                  **kwargs)
            for _ in range(depth)
        ])
        self.norm = QLayerNorm(embed_dim,
                               in1_bits=self.nof_bits_lnorm1,
                               in2_bits=self.nof_bits_lnorm2,
                               quant=self.quant)

        self.head = QuantizedLinear(embed_dim,
                                    num_classes,
                                    nof_bits1=8,
                                    nof_bits2=8,
                                    quant=self.quant) if num_classes > 0 else nn.Identity()
        self.head_dist = QuantizedLinear(embed_dim,
                                         num_classes,
                                         nof_bits1=8,
                                         nof_bits2=8,
                                         quant=self.quant) if distilled else None

        nn.init.trunc_normal_(self.pos_embed, std=0.02)
        nn.init.trunc_normal_(self.cls_token, std=0.02)
        if self.dist_token is not None:
            nn.init.trunc_normal_(self.dist_token, std=0.02)
        self.apply(self._init_weights)

    def _init_weights(self, m):
        if isinstance(m, nn.Linear) or isinstance(m, QuantizedLinear):
            nn.init.trunc_normal_(m.weight, std=0.02)
            if isinstance(m, nn.Linear) and m.bias is not None or isinstance(m, QuantizedLinear) and m.bias is not None:
                nn.init.zeros_(m.bias)
        elif isinstance(m, nn.LayerNorm) or isinstance(m, QLayerNorm):
            nn.init.ones_(m.weight)
            nn.init.zeros_(m.bias)

    def forward_features(self, x):
        B = x.shape[0]
        x = self.patch_embed(x)
        cls_tokens = self.cls_token.expand(B, -1, -1)  # Class token
        if self.dist_token is None:
            x = torch.cat((cls_tokens, x), dim=1)
        else:
            dist_token = self.dist_token.expand(B, -1, -1)  # Distillation token
            x = torch.cat((cls_tokens, dist_token, x), dim=1)
        x = self.pos_drop(x + self.pos_embed)
        for blk in self.blocks:
            x = blk(x)
        x = self.norm(x)
        if self.dist_token is None:
            return x[:, 0]  # return class token output
        else:
            return x[:, 0], x[:, 1]  # return class token and distillation token

    def forward(self, x):
        x = self.forward_features(x)
        if self.head_dist is not None:
            x_cls, x_dist = x
            x = self.head(x_cls), self.head_dist(x_dist)  # Class and distillation heads
            if not self.training:
                # during inference, return the average of both head outputs
                return (x[0] + x[1]) / 2
            else:
                return x
        else:
            x = self.head(x)
        return x



## Swin model

In [ ]:
def window_partition(x, window_size):
    """
    Args:
        x: (B, H, W, C)
        window_size (int): window size

    Returns:
        windows: (num_windows*B, window_size, window_size, C)
    """
    B, H, W, C = x.shape
    x = x.view(B, H // window_size, window_size, W // window_size, window_size, C)
    windows = x.permute(0, 1, 3, 2, 4, 5).contiguous().view(-1, window_size, window_size, C)
    return windows


def window_reverse(windows, window_size, H, W):
    """
    Args:
        windows: (num_windows*B, window_size, window_size, C)
        window_size (int): Window size
        H (int): Height of image
        W (int): Width of image

    Returns:
        x: (B, H, W, C)
    """
    B = int(windows.shape[0] / (H * W / window_size / window_size))
    x = windows.view(B, H // window_size, W // window_size, window_size, window_size, -1)
    x = x.permute(0, 1, 3, 2, 4, 5).contiguous().view(B, H, W, -1)
    return x




### window attention

In [ ]:
class WindowAttention(QauntParams):
    r""" Window based multi-head self attention (W-MSA) module with relative position bias.
    It supports both of shifted and non-shifted window.

    Args:
        dim (int): Number of input channels.
        window_size (tuple[int]): The height and width of the window.
        num_heads (int): Number of attention heads.
        qkv_bias (bool, optional):  If True, add a learnable bias to query, key, value. Default: True
        qk_scale (float | None, optional): Override default qk scale of head_dim ** -0.5 if set
        attn_drop (float, optional): Dropout ratio of attention weight. Default: 0.0
        proj_drop (float, optional): Dropout ratio of output. Default: 0.0
    """

    def __init__(self,
                 dim,
                 window_size,
                 num_heads,
                 qkv_bias=True,
                 qk_scale=None,
                 attn_drop=0.,
                 quant=False,
                 proj_drop=0.):

        super().__init__()
        self.quant = quant
        self.dim = dim
        self.window_size = window_size  # Wh, Ww
        self.num_heads = num_heads
        head_dim = dim // num_heads
        self.scale = qk_scale or head_dim ** -0.5

        # define a parameter table of relative position bias
        self.relative_position_bias_table = nn.Parameter(
            torch.zeros((2 * window_size[0] - 1) * (2 * window_size[1] - 1), num_heads))  # 2*Wh-1 * 2*Ww-1, nH

        # get pair-wise relative position index for each token inside the window
        coords_h = torch.arange(self.window_size[0])
        coords_w = torch.arange(self.window_size[1])
        coords = torch.stack(torch.meshgrid([coords_h, coords_w]))  # 2, Wh, Ww
        coords_flatten = torch.flatten(coords, 1)  # 2, Wh*Ww
        relative_coords = coords_flatten[:, :, None] - coords_flatten[:, None, :]  # 2, Wh*Ww, Wh*Ww
        relative_coords = relative_coords.permute(1, 2, 0).contiguous()  # Wh*Ww, Wh*Ww, 2
        relative_coords[:, :, 0] += self.window_size[0] - 1  # shift to start from 0
        relative_coords[:, :, 1] += self.window_size[1] - 1
        relative_coords[:, :, 0] *= 2 * self.window_size[1] - 1
        relative_position_index = relative_coords.sum(-1)  # Wh*Ww, Wh*Ww
        self.register_buffer("relative_position_index", relative_position_index)

        self.qkv = QuantizedLinear(dim,
                            dim * 3,
                            bias=qkv_bias,
                            nof_bits1=8,
                            nof_bits2=8,
                            quant=quant)

        self.mat_mul_qk = QuantizedMatmul(in1_bits=8,
                                  in2_bits=8,
                                  quant=quant)

        self.attn_drop = nn.Dropout(attn_drop)

        self.mat_mul_pv = QuantizedMatmul(in1_bits=8,
                                          in2_bits=8,
                                          quant=quant)

        self.proj = QuantizedLinear(dim,
                                    dim,
                                    nof_bits1=8,
                                    nof_bits2=8,
                                    quant=quant)

        self.proj_drop = nn.Dropout(proj_drop)

        trunc_normal_(self.relative_position_bias_table, std=.02)

        self.softmax = IntSoftmaxTS(nof_bits=8,
                               LUT_SIZE=16,
                               dim=-1,
                               quant=quant)

    def forward(self, x, mask=None):
        """
        Args:
            x: input features with shape of (num_windows*B, N, C)
            mask: (0/-inf) mask with shape of (num_windows, Wh*Ww, Wh*Ww) or None
        """
        B_, N, C = x.shape
        qkv = self.qkv(x).reshape(B_, N, 3, self.num_heads, C // self.num_heads).permute(2, 0, 3, 1, 4)
        q, k, v = qkv[0], qkv[1], qkv[2]  # make torchscript happy (cannot use tensor as tuple)

        attn = self.mat_mul_qk(q, k.transpose(-2, -1)) * self.scale  # Scaled dot-product attention

        relative_position_bias = self.relative_position_bias_table[self.relative_position_index.view(-1)].view(
            self.window_size[0] * self.window_size[1], self.window_size[0] * self.window_size[1], -1)  # Wh*Ww,Wh*Ww,nH
        relative_position_bias = relative_position_bias.permute(2, 0, 1).contiguous()  # nH, Wh*Ww, Wh*Ww
        attn = attn + relative_position_bias.unsqueeze(0)

        if mask is not None:
            nW = mask.shape[0]
            attn = attn.view(B_ // nW, nW, self.num_heads, N, N) + mask.unsqueeze(1).unsqueeze(0)
            attn = attn.view(-1, self.num_heads, N, N)
            attn = self.softmax(attn)
        else:
            attn = self.softmax(attn)

        attn = self.attn_drop(attn)

        # x = (attn @ v).transpose(1, 2).reshape(B_, N, C)
        x = self.mat_mul_pv(attn, v).transpose(1, 2).reshape(B_, N, C)

        x = self.proj(x)
        x = self.proj_drop(x)
        return x

    def extra_repr(self) -> str:
        return f'dim={self.dim}, window_size={self.window_size}, num_heads={self.num_heads}'

    def flops(self, N):
        # calculate flops for 1 window with token length of N
        flops = 0
        # qkv = self.qkv(x)
        flops += N * self.dim * 3 * self.dim
        # attn = (q @ k.transpose(-2, -1))
        flops += self.num_heads * N * (self.dim // self.num_heads) * N
        #  x = (attn @ v)
        flops += self.num_heads * N * N * (self.dim // self.num_heads)
        # x = self.proj(x)
        flops += N * self.dim * self.dim
        return flops




### swin transformer block

In [ ]:
class SwinTransformerBlock(QauntParams):
    r""" Swin Transformer Block.

    Args:
        dim (int): Number of input channels.
        input_resolution (tuple[int]): Input resulotion.
        num_heads (int): Number of attention heads.
        window_size (int): Window size.
        shift_size (int): Shift size for SW-MSA.
        mlp_ratio (float): Ratio of mlp hidden dim to embedding dim.
        qkv_bias (bool, optional): If True, add a learnable bias to query, key, value. Default: True
        qk_scale (float | None, optional): Override default qk scale of head_dim ** -0.5 if set.
        drop (float, optional): Dropout rate. Default: 0.0
        attn_drop (float, optional): Attention dropout rate. Default: 0.0
        drop_path (float, optional): Stochastic depth rate. Default: 0.0
        act_layer (nn.Module, optional): Activation layer. Default: nn.GELU
        norm_layer (nn.Module, optional): Normalization layer.  Default: nn.LayerNorm
        fused_window_process (bool, optional): If True, use one kernel to fused window shift & window partition for acceleration, similar for the reversed part. Default: False
    """

    def __init__(self,
                 dim,
                 input_resolution,
                 num_heads,
                 window_size=7,
                 shift_size=0,
                 mlp_ratio=4.,
                 qkv_bias=True,
                 qk_scale=None,
                 drop=0.,
                 attn_drop=0.,
                 drop_path=0.,
                 act_layer=IntGeluTS,
                 norm_layer=QLayerNorm,
                 quant=False,
                 fused_window_process=False):

        super().__init__()
        self.quant = quant
        self.dim = dim
        self.input_resolution = input_resolution
        self.num_heads = num_heads
        self.window_size = window_size
        self.shift_size = shift_size
        self.mlp_ratio = mlp_ratio
        if min(self.input_resolution) <= self.window_size:
            # if window size is larger than input resolution, we don't partition windows
            self.shift_size = 0
            self.window_size = min(self.input_resolution)
        assert 0 <= self.shift_size < self.window_size, "shift_size must in 0-window_size"

        # self.norm1 = norm_layer(dim)
        self.norm1 = norm_layer(dim,
                        in1_bits=self.nof_bits_lnorm1,
                        in2_bits=self.nof_bits_lnorm2,
                        quant=quant)

        self.attn = WindowAttention(dim,
                                    window_size=to_2tuple(self.window_size),
                                    num_heads=num_heads,
                                    qkv_bias=qkv_bias,
                                    qk_scale=qk_scale,
                                    attn_drop=attn_drop,
                                    quant=quant,
                                    proj_drop=drop)

        self.drop_path = DropPath(drop_path) if drop_path > 0. else nn.Identity()

        self.norm2 = norm_layer(dim,
                                in1_bits=self.nof_bits_lnorm1,
                                in2_bits=self.nof_bits_lnorm2,
                                quant=quant)

        mlp_hidden_dim = int(dim * mlp_ratio)
        self.mlp = Mlp(in_features=dim,
                       hidden_features=mlp_hidden_dim,
                       act_layer=act_layer,
                       drop=drop,
                       quant=quant)


        if self.shift_size > 0:
            # calculate attention mask for SW-MSA
            H, W = self.input_resolution
            img_mask = torch.zeros((1, H, W, 1))  # 1 H W 1
            h_slices = (slice(0, -self.window_size),
                        slice(-self.window_size, -self.shift_size),
                        slice(-self.shift_size, None))
            w_slices = (slice(0, -self.window_size),
                        slice(-self.window_size, -self.shift_size),
                        slice(-self.shift_size, None))
            cnt = 0
            for h in h_slices:
                for w in w_slices:
                    img_mask[:, h, w, :] = cnt
                    cnt += 1

            mask_windows = window_partition(img_mask, self.window_size)  # nW, window_size, window_size, 1
            mask_windows = mask_windows.view(-1, self.window_size * self.window_size)
            attn_mask = mask_windows.unsqueeze(1) - mask_windows.unsqueeze(2)
            attn_mask = attn_mask.masked_fill(attn_mask != 0, float(-100.0)).masked_fill(attn_mask == 0, float(0.0))
        else:
            attn_mask = None

        self.register_buffer("attn_mask", attn_mask)
        self.fused_window_process = fused_window_process

    def forward(self, x):
        H, W = self.input_resolution
        B, L, C = x.shape
        assert L == H * W, "input feature has wrong size"

        shortcut = x
        x = self.norm1(x)
        x = x.view(B, H, W, C)

        # cyclic shift
        if self.shift_size > 0:
            if not self.fused_window_process:
                shifted_x = torch.roll(x, shifts=(-self.shift_size, -self.shift_size), dims=(1, 2))
                # partition windows
                x_windows = window_partition(shifted_x, self.window_size)  # nW*B, window_size, window_size, C
            else:
                x_windows = WindowProcess.apply(x, B, H, W, C, -self.shift_size, self.window_size)
        else:
            shifted_x = x
            # partition windows
            x_windows = window_partition(shifted_x, self.window_size)  # nW*B, window_size, window_size, C

        x_windows = x_windows.view(-1, self.window_size * self.window_size, C)  # nW*B, window_size*window_size, C

        # W-MSA/SW-MSA
        attn_windows = self.attn(x_windows, mask=self.attn_mask)  # nW*B, window_size*window_size, C

        # merge windows
        attn_windows = attn_windows.view(-1, self.window_size, self.window_size, C)

        # reverse cyclic shift
        if self.shift_size > 0:
            if not self.fused_window_process:
                shifted_x = window_reverse(attn_windows, self.window_size, H, W)  # B H' W' C
                x = torch.roll(shifted_x, shifts=(self.shift_size, self.shift_size), dims=(1, 2))
            else:
                x = WindowProcessReverse.apply(attn_windows, B, H, W, C, self.shift_size, self.window_size)
        else:
            shifted_x = window_reverse(attn_windows, self.window_size, H, W)  # B H' W' C
            x = shifted_x
        x = x.view(B, H * W, C)
        x = shortcut + self.drop_path(x)

        # FFN
        x = x + self.drop_path(self.mlp(self.norm2(x)))

        return x

    def extra_repr(self) -> str:
        return f"dim={self.dim}, input_resolution={self.input_resolution}, num_heads={self.num_heads}, " \
               f"window_size={self.window_size}, shift_size={self.shift_size}, mlp_ratio={self.mlp_ratio}"

    def flops(self):
        flops = 0
        H, W = self.input_resolution
        # norm1
        flops += self.dim * H * W
        # W-MSA/SW-MSA
        nW = H * W / self.window_size / self.window_size
        flops += nW * self.attn.flops(self.window_size * self.window_size)
        # mlp
        flops += 2 * H * W * self.dim * self.dim * self.mlp_ratio
        # norm2
        flops += self.dim * H * W
        return flops



### PatchMerging

In [ ]:
class PatchMerging(QauntParams):
    r""" Patch Merging Layer.

    Args:
        input_resolution (tuple[int]): Resolution of input feature.
        dim (int): Number of input channels.
        norm_layer (nn.Module, optional): Normalization layer.  Default: nn.LayerNorm
    """

    def __init__(self,
                 input_resolution,
                 dim,
                 quant=False,
                 norm_layer=QLayerNorm):

        super().__init__()
        self.quant = quant
        self.input_resolution = input_resolution
        self.dim = dim
        # self.reduction = nn.Linear(4 * dim, 2 * dim, bias=False)
        self.reduction = QuantizedLinear(4 * dim, 2 * dim, bias=False)

        self.norm = norm_layer(4 * dim,
                                in1_bits=self.nof_bits_lnorm1,
                                in2_bits=self.nof_bits_lnorm2,
                                quant=quant)

    def forward(self, x):
        """
        x: B, H*W, C
        """
        H, W = self.input_resolution
        B, L, C = x.shape
        assert L == H * W, "input feature has wrong size"
        assert H % 2 == 0 and W % 2 == 0, f"x size ({H}*{W}) are not even."

        x = x.view(B, H, W, C)

        x0 = x[:, 0::2, 0::2, :]  # B H/2 W/2 C
        x1 = x[:, 1::2, 0::2, :]  # B H/2 W/2 C
        x2 = x[:, 0::2, 1::2, :]  # B H/2 W/2 C
        x3 = x[:, 1::2, 1::2, :]  # B H/2 W/2 C
        x = torch.cat([x0, x1, x2, x3], -1)  # B H/2 W/2 4*C
        x = x.view(B, -1, 4 * C)  # B H/2*W/2 4*C

        x = self.norm(x)
        x = self.reduction(x)

        return x

    def extra_repr(self) -> str:
        return f"input_resolution={self.input_resolution}, dim={self.dim}"

    def flops(self):
        H, W = self.input_resolution
        flops = H * W * self.dim
        flops += (H // 2) * (W // 2) * 4 * self.dim * 2 * self.dim
        return flops


### SWIN basic layer

In [ ]:
class BasicLayer(QauntParams):
    """ A basic Swin Transformer layer for one stage.

    Args:
        dim (int): Number of input channels.
        input_resolution (tuple[int]): Input resolution.
        depth (int): Number of blocks.
        num_heads (int): Number of attention heads.
        window_size (int): Local window size.
        mlp_ratio (float): Ratio of mlp hidden dim to embedding dim.
        qkv_bias (bool, optional): If True, add a learnable bias to query, key, value. Default: True
        qk_scale (float | None, optional): Override default qk scale of head_dim ** -0.5 if set.
        drop (float, optional): Dropout rate. Default: 0.0
        attn_drop (float, optional): Attention dropout rate. Default: 0.0
        drop_path (float | tuple[float], optional): Stochastic depth rate. Default: 0.0
        norm_layer (nn.Module, optional): Normalization layer. Default: nn.LayerNorm
        downsample (nn.Module | None, optional): Downsample layer at the end of the layer. Default: None
        use_checkpoint (bool): Whether to use checkpointing to save memory. Default: False.
        fused_window_process (bool, optional): If True, use one kernel to fused window shift & window partition for acceleration, similar for the reversed part. Default: False
    """

    def __init__(self,
                 dim,
                 input_resolution,
                 depth,
                 num_heads,
                 window_size,
                 mlp_ratio=4.,
                 qkv_bias=True,
                 qk_scale=None,
                 drop=0.,
                 attn_drop=0.,
                 drop_path=0.,
                 norm_layer=QLayerNorm,
                 downsample=None,
                 use_checkpoint=False,
                 quant=False,
                 fused_window_process=False):

        super().__init__()
        self.quant = quant
        self.dim = dim
        self.input_resolution = input_resolution
        self.depth = depth
        self.use_checkpoint = use_checkpoint

        # build blocks
        self.blocks = nn.ModuleList([
            SwinTransformerBlock(dim=dim, input_resolution=input_resolution,
                                 act_layer=IntGeluTS,
                                 num_heads=num_heads, window_size=window_size,
                                 shift_size=0 if (i % 2 == 0) else window_size // 2,
                                 mlp_ratio=mlp_ratio,
                                 qkv_bias=qkv_bias, qk_scale=qk_scale,
                                 drop=drop, attn_drop=attn_drop,
                                 drop_path=drop_path[i] if isinstance(drop_path, list) else drop_path,
                                 norm_layer=norm_layer,
                                 quant=quant,
                                 fused_window_process=fused_window_process)
            for i in range(depth)])

        # patch merging layer
        if downsample is not None:
            self.downsample = downsample(input_resolution, dim=dim, norm_layer=norm_layer)
        else:
            self.downsample = None

    def forward(self, x):
        for blk in self.blocks:
            if self.use_checkpoint:
                x = checkpoint.checkpoint(blk, x)
            else:
                x = blk(x)
        if self.downsample is not None:
            x = self.downsample(x)
        return x

    def extra_repr(self) -> str:
        return f"dim={self.dim}, input_resolution={self.input_resolution}, depth={self.depth}"

    def flops(self):
        flops = 0
        for blk in self.blocks:
            flops += blk.flops()
        if self.downsample is not None:
            flops += self.downsample.flops()
        return flops


### PatchEmbed

In [ ]:
class PatchEmbed(QauntParams):
    r""" Image to Patch Embedding

    Args:
        img_size (int): Image size.  Default: 224.
        patch_size (int): Patch token size. Default: 4.
        in_chans (int): Number of input image channels. Default: 3.
        embed_dim (int): Number of linear projection output channels. Default: 96.
        norm_layer (nn.Module, optional): Normalization layer. Default: None
    """

    def __init__(self,
                 img_size=224,
                 patch_size=4,
                 in_chans=3,
                 embed_dim=96,
                 quant=False,
                 norm_layer=None):

        super().__init__()
        self.quant = quant
        img_size = to_2tuple(img_size)
        patch_size = to_2tuple(patch_size)
        patches_resolution = [img_size[0] // patch_size[0], img_size[1] // patch_size[1]]
        self.img_size = img_size
        self.patch_size = patch_size
        self.patches_resolution = patches_resolution
        self.num_patches = patches_resolution[0] * patches_resolution[1]

        self.in_chans = in_chans
        self.embed_dim = embed_dim

        self.proj = nn.Conv2d(in_chans, embed_dim, kernel_size=patch_size, stride=patch_size)
        if norm_layer is not None:
            self.norm = norm_layer(embed_dim,
                                    in1_bits=self.nof_bits_lnorm1,
                                    in2_bits=self.nof_bits_lnorm2,
                                    quant=quant)
        else:
            self.norm = None

    def forward(self, x):
        B, C, H, W = x.shape
        # FIXME look at relaxing size constraints
        # assert H == self.img_size[0] and W == self.img_size[1], \
        #     f"Input image size ({H}*{W}) doesn't match model ({self.img_size[0]}*{self.img_size[1]})."
        x = self.proj(x).flatten(2).transpose(1, 2)  # B Ph*Pw C
        if self.norm is not None:
            x = self.norm(x)
        return x

    def flops(self):
        Ho, Wo = self.patches_resolution
        flops = Ho * Wo * self.embed_dim * self.in_chans * (self.patch_size[0] * self.patch_size[1])
        if self.norm is not None:
            flops += Ho * Wo * self.embed_dim
        return flops




### swin transformer

In [ ]:
class SwinTransformer(QuantTransformer):
    r""" Swin Transformer
        A PyTorch impl of : `Swin Transformer: Hierarchical Vision Transformer using Shifted Windows`  -
          https://arxiv.org/pdf/2103.14030

    Args:
        img_size (int | tuple(int)): Input image size. Default 224
        patch_size (int | tuple(int)): Patch size. Default: 4
        in_chans (int): Number of input image channels. Default: 3
        num_classes (int): Number of classes for classification head. Default: 1000
        embed_dim (int): Patch embedding dimension. Default: 96
        depths (tuple(int)): Depth of each Swin Transformer layer.
        num_heads (tuple(int)): Number of attention heads in different layers.
        window_size (int): Window size. Default: 7
        mlp_ratio (float): Ratio of mlp hidden dim to embedding dim. Default: 4
        qkv_bias (bool): If True, add a learnable bias to query, key, value. Default: True
        qk_scale (float): Override default qk scale of head_dim ** -0.5 if set. Default: None
        drop_rate (float): Dropout rate. Default: 0
        attn_drop_rate (float): Attention dropout rate. Default: 0
        drop_path_rate (float): Stochastic depth rate. Default: 0.1
        norm_layer (nn.Module): Normalization layer. Default: nn.LayerNorm.
        ape (bool): If True, add absolute position embedding to the patch embedding. Default: False
        patch_norm (bool): If True, add normalization after patch embedding. Default: True
        use_checkpoint (bool): Whether to use checkpointing to save memory. Default: False
        fused_window_process (bool, optional): If True, use one kernel to fused window shift & window partition for acceleration, similar for the reversed part. Default: False
    """

    def __init__(self,
                 img_size=224,
                 patch_size=4,
                 in_chans=3,
                 num_classes=1000,
                 embed_dim=96,
                 depths=[2, 2, 6, 2],
                 num_heads=[3, 6, 12, 24],
                 window_size=7,
                 mlp_ratio=4.,
                 qkv_bias=True,
                 qk_scale=None,
                 drop_rate=0.,
                 attn_drop_rate=0.,
                 drop_path_rate=0.1,
                 norm_layer=QLayerNorm,
                 ape=False,
                 patch_norm=True,
                 is_calibrate=False,
                 quant=False,
                 use_checkpoint=False,
                 fused_window_process=False,
                 q_module_list=[QuantizedLinear, QuantizedMatmul, qHadamardProd, QLayerNorm], **kwargs):
        super().__init__(
            quant=quant,
            is_calibrate=is_calibrate,
            q_module_list=q_module_list
        )
        self.quant = quant
        self.num_classes = num_classes
        self.num_layers = len(depths)
        self.embed_dim = embed_dim
        self.ape = ape
        self.patch_norm = patch_norm
        self.num_features = int(embed_dim * 2 ** (self.num_layers - 1))
        self.mlp_ratio = mlp_ratio
        self.q_module_list = q_module_list

        # split image into non-overlapping patches
        self.patch_embed = PatchEmbed(
            img_size=img_size,
            patch_size=patch_size,
            in_chans=in_chans,
            embed_dim=embed_dim,
            quant=quant,
            norm_layer=norm_layer if self.patch_norm else None)

        num_patches = self.patch_embed.num_patches
        patches_resolution = self.patch_embed.patches_resolution
        self.patches_resolution = patches_resolution

        # absolute position embedding
        if self.ape:
            self.absolute_pos_embed = nn.Parameter(torch.zeros(1, num_patches, embed_dim))
            trunc_normal_(self.absolute_pos_embed, std=.02)

        self.pos_drop = nn.Dropout(p=drop_rate)

        # stochastic depth
        dpr = [x.item() for x in torch.linspace(0, drop_path_rate, sum(depths))]  # stochastic depth decay rule

        # build layers
        self.layers = nn.ModuleList()
        for i_layer in range(self.num_layers):
            layer = BasicLayer(dim=int(embed_dim * 2 ** i_layer),
                               input_resolution=(patches_resolution[0] // (2 ** i_layer),
                                                 patches_resolution[1] // (2 ** i_layer)),
                               depth=depths[i_layer],
                               num_heads=num_heads[i_layer],
                               window_size=window_size,
                               mlp_ratio=self.mlp_ratio,
                               qkv_bias=qkv_bias, qk_scale=qk_scale,
                               drop=drop_rate, attn_drop=attn_drop_rate,
                               drop_path=dpr[sum(depths[:i_layer]):sum(depths[:i_layer + 1])],
                               norm_layer=norm_layer,
                               downsample=PatchMerging if (i_layer < self.num_layers - 1) else None,
                               use_checkpoint=use_checkpoint,
                               quant=quant,
                               fused_window_process=fused_window_process)
            self.layers.append(layer)

        self.norm = norm_layer(self.num_features,
                               in1_bits=self.nof_bits_lnorm1,
                               in2_bits=self.nof_bits_lnorm2)
        self.avgpool = nn.AdaptiveAvgPool1d(1)
        self.head = nn.Linear(self.num_features, num_classes) if num_classes > 0 else nn.Identity()

        self.apply(self._init_weights)

    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            trunc_normal_(m.weight, std=.02)
            if isinstance(m, nn.Linear) and m.bias is not None:
                nn.init.constant_(m.bias, 0)
        elif isinstance(m, nn.LayerNorm):
            nn.init.constant_(m.bias, 0)
            nn.init.constant_(m.weight, 1.0)

    @torch.jit.ignore
    def no_weight_decay(self):
        return {'absolute_pos_embed'}

    @torch.jit.ignore
    def no_weight_decay_keywords(self):
        return {'relative_position_bias_table'}

    def forward_features(self, x):
        x = self.patch_embed(x)
        if self.ape:
            x = x + self.absolute_pos_embed
        x = self.pos_drop(x)

        for layer in self.layers:
            x = layer(x)

        x = self.norm(x)  # B L C
        x = self.avgpool(x.transpose(1, 2))  # B C 1
        x = torch.flatten(x, 1)
        return x

    def forward(self, x):
        x = self.forward_features(x)
        x = self.head(x)
        return x

    def flops(self):
        flops = 0
        flops += self.patch_embed.flops()
        for i, layer in enumerate(self.layers):
            flops += layer.flops()
        flops += self.num_features * self.patches_resolution[0] * self.patches_resolution[1] // (2 ** self.num_layers)
        flops += self.num_features * self.num_classes
        return flops



# load model functions

In [ ]:
def deit_base_patch16_224(pretrained=False, quant=False, q_module_list=[], **kwargs):
    model = DistilledVisionTransformer(
        img_size=224,
        patch_size=16,
        embed_dim=768,
        depth=12,
        num_heads=12,
        mlp_ratio=4,
        qkv_bias=True,
        quant=quant,
        q_module_list=q_module_list,
        distilled=False,  # Disable distillation
        **kwargs)
    if pretrained:
        checkpoint = torch.hub.load_state_dict_from_url(
            'https://dl.fbaipublicfiles.com/deit/deit_base_patch16_224-b5f2ef4d.pth', map_location='cpu')
        model.load_state_dict(checkpoint['model'])
    return model

def deit_small_patch16_224(pretrained=False, quant=False, q_module_list=[], **kwargs):
    """
    DeiT-Small (DeiT-S) with patch size of 16 and input image size of 224x224.
    """
    model = DistilledVisionTransformer(
        img_size=224,
        patch_size=16,
        embed_dim=384,
        depth=12,
        num_heads=6,
        mlp_ratio=4,
        qkv_bias=True,
        q_module_list=q_module_list,
        quant=quant,
        distilled=False,  # Disable distillation
        **kwargs)

    if pretrained:
        checkpoint = torch.hub.load_state_dict_from_url(
            'https://dl.fbaipublicfiles.com/deit/deit_small_patch16_224-cd65a155.pth', map_location='cpu')
        model.load_state_dict(checkpoint['model'])
    return model

def deit_tiny_patch16_224(pretrained=False, quant=False, q_module_list=[], **kwargs):
    """
    DeiT-Tiny (DeiT-T) with patch size of 16 and input image size of 224x224.
    """
    model = DistilledVisionTransformer(
        img_size=224,
        patch_size=16,
        embed_dim=192,
        depth=12,
        num_heads=3,
        mlp_ratio=4,
        qkv_bias=True,
        q_module_list=q_module_list,
        quant=quant,
        distilled=False,  # Disable distillation
        **kwargs)
    if pretrained:
        checkpoint = torch.hub.load_state_dict_from_url(
            'https://dl.fbaipublicfiles.com/deit/deit_tiny_patch16_224-a1311bcf.pth', map_location='cpu')
        model.load_state_dict(checkpoint['model'])
    return model

def swin_tiny_patch4_window7_224(pretrained=False, quant=False, q_module_list=[], **kwargs):
    model = SwinTransformer(
        img_size=224,
        patch_size=4,
        num_classes=1000,
        quant=quant,
        norm_layer=QLayerNorm,
        embed_dim=96,
        depths=(2, 2, 6, 2),
        num_heads=(3, 6, 12, 24),
        window_size=7,  # Set as an integer
        q_module_list=q_module_list,
        **kwargs
    )

    if pretrained:
      # Load the checkpoint with strict=False to ignore any minor mismatches
      checkpoint = torch.hub.load_state_dict_from_url(
          'https://github.com/SwinTransformer/storage/releases/download/v1.0.0/swin_tiny_patch4_window7_224.pth'
      )
      model.load_state_dict(checkpoint['model'], strict=False)

    return model

def swin_small_patch4_window7_224(pretrained=False, quant=False, q_module_list=[], **kwargs):
    model = SwinTransformer(
        img_size=224,
        patch_size=4,
        num_classes=1000,
        norm_layer=QLayerNorm,
        embed_dim=96,
        quant=quant,
        depths=(2, 2, 18, 2),
        num_heads=(3, 6, 12, 24),
        window_size=7,  # Set as an integer
        q_module_list=q_module_list,
        **kwargs
    )

    if pretrained:
      # Load the checkpoint with strict=False to ignore any minor mismatches
      checkpoint = torch.hub.load_state_dict_from_url(
          'https://github.com/SwinTransformer/storage/releases/download/v1.0.0/swin_small_patch4_window7_224.pth'
      )
      model.load_state_dict(checkpoint['model'], strict=False)

    return model

def swin_base_patch4_window7_224(pretrained=False, quant=False, q_module_list=[], **kwargs):
    model = SwinTransformer(
        img_size=224,
        patch_size=4,
        num_classes=1000,
        norm_layer=QLayerNorm,
        quant=quant,
        embed_dim=128,  # Base embedding dimension for swin_base
        depths=[2, 2, 18, 2],
        num_heads=[4, 8, 16, 32],  # Correct head configuration
        window_size=7,  # Set as an integer
        q_module_list=q_module_list,
        **kwargs
    )

    if pretrained:
      # Load the checkpoint with strict=False to ignore any minor mismatches
      checkpoint = torch.hub.load_state_dict_from_url(
          'https://github.com/SwinTransformer/storage/releases/download/v1.0.0/swin_base_patch4_window7_224.pth'
      )
      model.load_state_dict(checkpoint['model'], strict=False)

    return model


# Initialize model

### Constractor

In [ ]:
# q_module_list=[QuantizedLinear, QuantizedMatmul, IntSoftmaxTS, QLayerNorm, IntGeluTS, qHadamardProd]
# q_module_list=[IntGeluTS, QuantizedLinear, QuantizedMatmul, IntSoftmaxTS, qHadamardProd, QLayerNorm]
q_module_list=[QLayerNorm]
# q_module_list=[IntSoftmaxTS]
# q_module_list=[IntGeluTS]

quant = False
batch_size = 8

model_to_run = {"deit_tiny_patch16_224" : deit_tiny_patch16_224,
              "deit_small_patch16_224" : deit_small_patch16_224,
              "deit_base_patch16_224" : deit_base_patch16_224,
              "swin_tiny_patch4_window7_224" : swin_tiny_patch4_window7_224,
              "swin_small_patch4_window7_224" : swin_small_patch4_window7_224,
              "swin_base_patch4_window7_224" : swin_base_patch4_window7_224}

model_name = "deit_tiny_patch16_224"
model = model_to_run[model_name](pretrained=True, q_module_list=q_module_list, quant=quant)
if torch.cuda.is_available():
    device = torch.device("cuda")  # Use GPU
    print("GPU is available.")
else:
    device = torch.device("cpu")  # Use CPU
    print("GPU is not available, using CPU.")
model = model.to(device)
print(model.eval())  # Set the model to evaluation mode
model.set_quant()  # set quantized block from a list (insead of quant=True)

Downloading: "https://dl.fbaipublicfiles.com/deit/deit_tiny_patch16_224-a1311bcf.pth" to /root/.cache/torch/hub/checkpoints/deit_tiny_patch16_224-a1311bcf.pth


100%|██████████| 21.9M/21.9M [00:00<00:00, 125MB/s] 


GPU is available.
DistilledVisionTransformer(
  (patch_embed): PatchEmbed(
    (proj): Conv2d(3, 192, kernel_size=(16, 16), stride=(16, 16))
  )
  (pos_drop): Dropout(p=0.1, inplace=False)
  (blocks): ModuleList(
    (0-11): 12 x Block(
      (norm1): QLayerNorm(
        (192,), eps=1e-05, elementwise_affine=True
        (in_obs_normalize): MinMaxObserver()
        (in_obs): MinMaxObserver()
        (w_obs): MinMaxObserver()
        (b_obs): MinMaxObserver()
      )
      (attn): Attention(
        (qkv): QuantizedLinear(
          in_features=192, out_features=576, bias=True
          (in_obs): MinMaxObserver()
          (w_obs): MinMaxObserver()
          (b_obs): MinMaxObserver()
        )
        (mat_mul_qk): QuantizedMatmul(
          (in1_obs): MinMaxObserver()
          (in2_obs): MinMaxObserver()
          (out_obs): MinMaxObserver()
        )
        (sf): IntSoftmaxTS(
          (in_obs): MinMaxObserver()
        )
        (attn_drop): Dropout(p=0.0, inplace=False)
        (

### huggingface token

In [ ]:
# del hug_cli  # to regenerate the hug token
try:
  if hug_cli:
    pass
  else:
    raise NameError
except NameError as err:
  !hf auth login
  hug_cli=True


    _|    _|  _|    _|    _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|_|_|_|    _|_|      _|_|_|  _|_|_|_|
    _|    _|  _|    _|  _|        _|          _|    _|_|    _|  _|            _|        _|    _|  _|        _|
    _|_|_|_|  _|    _|  _|  _|_|  _|  _|_|    _|    _|  _|  _|  _|  _|_|      _|_|_|    _|_|_|_|  _|        _|_|_|
    _|    _|  _|    _|  _|    _|  _|    _|    _|    _|    _|_|  _|    _|      _|        _|    _|  _|        _|
    _|    _|    _|_|      _|_|_|    _|_|_|  _|_|_|  _|      _|    _|_|_|      _|        _|    _|    _|_|_|  _|_|_|_|

    To log in, `huggingface_hub` requires a token generated from https://huggingface.co/settings/tokens .
Enter your token (input will not be visible): 

### set data loaders

In [ ]:
import math
load_dataset_flag = 0
try:
    if load_dataset_flag:
        pass
    else:
        raise NameError
except NameError as err:

        # Create stream set for validation and calibration
        print("loading calibration dataset (huggingface server)...")
        dset_calib = load_dataset('imagenet-1k',split='train',
                                  streaming=True,
                                  token=True,
                                  trust_remote_code=True
        #                          ).shuffle(seed=42).take(100)
                                 ).shuffle(seed=42).take(100)
        print("done")

        print("loading imagenet-1k test dataset from huggingface server...")
        dset_infer = load_dataset('imagenet-1k',split='validation',
                                  streaming=True,
                                  token=True,
                                  trust_remote_code=True
        #                          ).shuffle(seed=42).take(100)
                                 ).shuffle(seed=42)
        print("done")

        print("loading scale optimization dataset (huggingface server)...")
        dset_scale_opt = load_dataset('imagenet-1k',split='train',
                                  streaming=True,
                                  token=True,
                                  trust_remote_code=True
                                 ).shuffle(seed=12).take(100)
        print("done")

        # Image preprocessing
        # Define image preprocessing
        # Ensure the image is resized to 224x224 which is compatible with patch size 4 and window size 7
        preprocess = transforms.Compose([
            transforms.Lambda(lambda img: img.convert("RGB") if img.mode != "RGB" else img),  # Convert grayscale to RGB
            transforms.Resize(256,  interpolation=transforms.InterpolationMode.BICUBIC),
            transforms.CenterCrop(224),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ])

        def fit_image(tst):
            # "They change it to binary. I don't know what the hell they were thinking", some version would need the below line
            # tst['image'] = Image.open(io.BytesIO(tst['image']['bytes']))
            tst['image'] = preprocess(tst['image'])
            tst['label'] = tst['label']
            return tst

        dset_infer_updated     = dset_infer.map(fit_image)
        dset_calib_updated     = dset_calib.map(fit_image)
        dset_scale_opt_updated = dset_scale_opt.map(fit_image)

        print("Building a calibration loader...")
        val_loader   = dset_infer_updated.iter(batch_size=batch_size)
        train_loader = dset_calib_updated.iter(batch_size=1)
        scale_opt_loader = dset_scale_opt_updated.iter(batch_size=1)

        it = iter(dset_calib) # for one image inference
        load_dataset_flag = 1

        print("done")


loading calibration dataset (huggingface server)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


LocalTokenNotFoundError: Token is required (`token=True`), but no token found. You need to provide a token or be logged in to Hugging Face with `hf auth login` or `huggingface_hub.login`. See https://huggingface.co/settings/tokens.

### clibrate model

In [ ]:

# del image_list  # recreate a new dataset
try:
    if image_list:
        pass
    else:
        raise NameError
except NameError as err:

    # Get N (args.calib_iter) labeled images for the calibration set.
    print("Building calibration set...")
    image_list = []
    for i, info in enumerate(train_loader):

        data = torch.stack(info['image'])
        data = data.to(device)

        if i == 32:
            print("break")
            break
        print(i, end="|")
        data = data.to(device)
        image_list.append(data)

    print("done")

    # Get N (args.calib_iter) labeled images for the calibration set.
    print("Building fine tuning set...")
    scale_opt_image_list = []
    for i, info in enumerate(scale_opt_loader):

        data = torch.stack(info['image'])
        data = data.to(device)

        if i == 32:
            print("break")
            break
        print(i, end="|")
        data = data.to(device)
        scale_opt_image_list.append(data)

    print("done")

    # Download ImageNet class index
    print("Download ImageNet class index")
    url = 'https://raw.githubusercontent.com/pytorch/hub/master/imagenet_classes.txt'
    response = urllib.request.urlopen(url)
    imagenet_classes = [line.strip() for line in response.readlines()]
    print("done")
    print("Calibration set is ready")

print("Quantizing model...")
# Use the stream data set to calibrate the quantized model
with torch.no_grad():
    model.set_calibration_flag()
    print()
    for i, image in enumerate(image_list):
        print(i, end="|")
        output = model(image)
    model.unset_calibration_flag()
    print("Quantization parameter were set")

print("All done")

In [ ]:

# # create a graph
# from torch.fx import symbolic_trace

# # Symbolic tracing frontend - captures the semantics of the module
# symbolic_traced: torch.fx.GraphModule = symbolic_trace(model.eval())

# # High-level intermediate representation (IR) - Graph representation
# print(symbolic_traced.graph)

# from torch.fx.passes.graph_drawer import FxGraphDrawer

# # symbolic_traced must be an fx.GraphModule (e.g., from fx.symbolic_trace or torch.export(...).graph_module)
# drawer = FxGraphDrawer(symbolic_traced, "fx_graph")   # <-- pass the GraphModule itself
# dot = drawer.get_dot_graph()                          # this is a pydot.Dot
# dot.write_svg("fx_graph.svg")                         # save visualization
# # or: dot.write_png("fx_graph.png"), dot.write_pdf("fx_graph.pdf")

# aksdljflkjaslkdf

## Optimizing input scale

In [ ]:
def cosine_similarity(tensor1, tensor2):
    """Calculates cosine similarity between two tensors."""
    tensor1 = tensor1.flatten()
    tensor2 = tensor2.flatten()
    dot_product = torch.dot(tensor1, tensor2)
    norm1 = torch.norm(tensor1)
    norm2 = torch.norm(tensor2)
    return dot_product / (norm1 * norm2)

def nrmse(y_true: torch.Tensor, y_pred: torch.Tensor, eps: float = 1e-8, norm: str = "var") -> torch.Tensor:
    """
    Compute Normalized Root Mean Squared Error (NRMSE).

    Args:
        y_true: Ground truth tensor.
        y_pred: Predicted tensor (same shape as y_true).
        eps: Small constant to avoid division by zero.
        norm: Normalization method:
              - "var": normalize by sqrt(Var(y_true))  (default, scale-invariant)
              - "minmax": normalize by (max(y_true) - min(y_true))
              - "mean": normalize by mean(|y_true|)
              - "none": no normalization (just RMSE)

    Returns:
        Scalar tensor with NRMSE value.
    """
    mse = torch.mean((y_true - y_pred) ** 2)
    rmse = torch.sqrt(mse)

    if norm == "var":
        denom = torch.sqrt(y_true.var(unbiased=False) + eps)
    elif norm == "minmax":
        denom = (y_true.max() - y_true.min()).clamp(min=eps)
    elif norm == "mean":
        denom = y_true.abs().mean().clamp(min=eps)
    elif norm == "none":
        denom = 1.0
    else:
        raise ValueError(f"Unknown normalization method: {norm}")

    return rmse / denom

In [ ]:
import numpy as np

def calc_cosine(layer, original_output, x, alpha=0.9, beta=1.1, num_scales=10):
    """
    Optimizes the scale factors for a layer normalization module using cosine similarity.

    Args:
        layer: The LayerNorm module whose scale factors will be optimized.
        original_output: A reference output for comparison.
        x: Input tensor to compute the cosine similarity.
        alpha: Lower bound for scale factor search.
        beta: Upper bound for scale factor search.
        num_scales: Number of scale factors to test within the [alpha, beta] range.

    Returns:
        None. Updates the layer with the optimal scale factors for weights and inputs.
    """
    # Optimize weight scale factor
    optimal_weight_scale = find_optimal_scale_factor(layer.update_w_scale_factor,
                                                     layer.forward_pass,
                                                     x,
                                                     original_output,
                                                     alpha, beta, num_scales)

    # Optimize input scale factor
    optimal_in_scale = find_optimal_scale_factor(layer.update_in_scale_factor,
                                                 layer.forward_pass,
                                                 x,
                                                 original_output,
                                                 alpha, beta, num_scales)

    # Apply the optimal scale factors to layer
    layer.set_optimal_w_scale_factor(optimal_weight_scale)
    layer.set_optimal_in_scale_factor(optimal_in_scale)


def find_optimal_scale_factor(update_scale_fn, forward_pass, x, original_output, alpha, beta, num_scales):
    """
    Finds the optimal scale factor that maximizes cosine similarity.

    Args:
        update_scale_fn: Function to update the scale factor.
        forward_pass: The forward pass function of the layer being optimized.
        x: Input tensor.
        original_output: The reference output for comparison.
        alpha: Lower bound for scale factor search.
        beta: Upper bound for scale factor search.
        num_scales: Number of scale factors to test.

    Returns:
        optimal_scale (float): The scale factor that gives the highest cosine similarity.
    """
    best_cosine = -float('inf')
    optimal_scale = 1.0

    # Test scale factors within the range [alpha, beta]
    for scale_factor in np.linspace(alpha, beta, num=num_scales):
        update_scale_fn(scale_factor)
        approx_output = forward_pass(x)
        cosine_sim = cosine_similarity(original_output, approx_output)

        if cosine_sim > best_cosine:
            best_cosine = cosine_sim
            optimal_scale = scale_factor

    return optimal_scale


In [ ]:
def update_scale_factor(mdl, obs, cur_min, cur_max, search_scale):
      obs.min_val = cur_min * search_scale
      obs.max_val = cur_max * search_scale
      obs.calculate_qparams()
      mdl.is_weights_quantized = False


def optimal_factor_search(x, mdl, obs, is_weight=False, alpha=0.5, betta=2.0, itr=10, cosine_similarity=cosine_similarity):

  # sample the original scale parameters
  cur_min = obs.min_val
  cur_max = obs.max_val

  # base cosine and the optimal scale
  optimal_cosine = -1
  optimal_scale  = 1

  for search_scale in np.linspace(alpha, betta, num=itr):

      update_scale_factor(mdl, obs, cur_min, cur_max, search_scale)

      # calc cosine similarity
      approx_output = mdl.forward_pass(x)
      original_output = mdl.float_forward_pass(x)
      cosine_sim = cosine_similarity(original_output, approx_output)

      if cosine_sim > optimal_cosine:
          optimal_cosine = cosine_sim
          optimal_scale = search_scale

  # revert changes
  update_scale_factor(mdl, obs, cur_min, cur_max, 1)

  return optimal_scale

def optimal_factor_search2(x1, x2, mdl, obs, is_weight=False, alpha=0.5, betta=2.0, itr=10):

  # sample the original scale parameters
  cur_min = obs.min_val
  cur_max = obs.max_val

  # base cosine and the optimal scale
  optimal_cosine = -1
  optimal_scale  = 1

  for search_scale in np.linspace(alpha, betta, num=itr):

      update_scale_factor(mdl, obs, cur_min, cur_max, search_scale)

      # calc cosine similarity
      approx_output = mdl.forward_pass(x1, x2)
      original_output = mdl.float_forward_pass(x1, x2)
      cosine_sim = cosine_similarity(original_output, approx_output)

      if cosine_sim > optimal_cosine:
          optimal_cosine = cosine_sim
          optimal_scale = search_scale

  # revert changes
  update_scale_factor(mdl, obs, cur_min, cur_max, 1)

  return optimal_scale

def optimal_factor_search_lnorm(mdl, obs, is_weight=False, alpha=0.5, betta=2.0, itr=10):

  # sample the original scale parameters
  cur_min = obs.min_val
  cur_max = obs.max_val

  # base cosine and the optimal scale
  optimal_cosine = 100
  optimal_scale  = 1

  print("_______________________________________________")
  for search_scale in np.linspace(alpha, betta, num=itr):

      update_scale_factor(mdl, obs, cur_min, cur_max, search_scale)
      weight_integer = obs.quantizer(mdl.weight)

      # calc cosine similarity
      cosine_sim = nrmse(obs.dequantizer(weight_integer), mdl.weight)

      print(f"search_scale:{search_scale}, optimal_cosine:{optimal_cosine}, cosine_sim:{cosine_sim}")
      if (optimal_cosine - cosine_sim) > 0.1:
          optimal_cosine = cosine_sim
          optimal_scale = search_scale
  print("_______________________________________________")

  # revert changes
  update_scale_factor(mdl, obs, cur_min, cur_max, 1)
  print("-----------------")
  return optimal_scale

print("optimizing model scale-factors...")
with torch.no_grad():

  model.set_scale_opt()
  print()
  for i, image in enumerate(scale_opt_image_list):
      print(i, end="|")
      output = model(image)
      for m in model.modules():
          if not type(m) in q_module_list:
              continue
          if type(m) in [QuantizedLinear]:
              x = m.opt_input
              dq_output = m.forward_pass(x)
              output = m.float_forward_pass(x)
              cs = cosine_similarity(output, dq_output)
              if cs < (1 - 0.01):
                  print("---------------")
                  print("QuantizedLinear: cosine similarity:", f"{float(cs):.2f}")
                  search_scale_w = optimal_factor_search(x, m, m.w_obs, is_weight=True, alpha=0.5, betta=2.0, itr=10)
                  update_scale_factor(m, m.w_obs, m.w_obs.min_val, m.w_obs.max_val,    search_scale_w)
                  search_scale_in = optimal_factor_search(x, m, m.in_obs, is_weight=True, alpha=0.5, betta=2.0, itr=10)
                  update_scale_factor(m, m.in_obs, m.in_obs.min_val, m.in_obs.max_val, search_scale_in)

          if type(m) in [QuantizedMatmul]:
              x1 = m.opt_input1
              x2 = m.opt_input2
              dq_output = m.forward_pass(x1, x2)
              output = m.float_forward_pass(x1, x2)
              cs = cosine_similarity(output, dq_output)
              if cs < (1 - 0.01):
                  print("---------------")
                  print("QuantizedMatmul: cosine similarity:", f"{float(cs):.2f}")
                  search_scale_in1 = optimal_factor_search2(x1, x2, m, m.in1_obs, is_weight=False, alpha=0.5, betta=2.0, itr=10)
                  update_scale_factor(m, m.in1_obs, m.in1_obs.min_val, m.in1_obs.max_val, search_scale_in1)
                  search_scale_in2 = optimal_factor_search2(x1, x2, m, m.in2_obs, is_weight=False, alpha=0.5, betta=2.0, itr=10)
                  update_scale_factor(m, m.in2_obs, m.in2_obs.min_val, m.in2_obs.max_val, search_scale_in2)

          if type(m) in [QLayerNorm] and i == 0:
              x = m.opt_input
              dq_output = m.forward_pass(x)
              cs = nrmse(m.w_obs.dequantizer(m.weight_integer), m.weight)

              mean_val = m.normalize(x)
              mean_val_f = m.float_foward_pass_normalize(x)
              dq_output = m.betta_gamma_forward_pass(mean_val)
              dq_output_f = m.float_betta_gamma_forward_pass(mean_val)
              # print(f"[QUANT] : {mean_val}")
              # print(f"[FLOAT32] : {mean_val_f}")
              # print("_________________________")
              # print(f"[QUANT] : {dq_output}")
              # print(f"[FLOAT32] : {dq_output_f}")

              # print("QLayerNorm: nrmse:", f"{float(cs):.2f}")
              # print(f"[QUANT] : {m.w_obs.dequantizer(m.weight_integer)}")
              # print(f"[FLOAT32] : {m.weight}")
              # if cs > (0.5):
              if cs > 0.03:
                  # print(f"[FLOAT32] : {m.weight_integer}")
                  pass
                  search_scale_w = optimal_factor_search_lnorm(m, m.w_obs, is_weight=False, alpha=0.5, betta=2.0, itr=50)
                  print("search_scale_w", search_scale_w)
                  update_scale_factor(m, m.w_obs, m.w_obs.min_val, m.w_obs.max_val, search_scale_w)

              print("---------------")



  model.unset_scale_opt()
  print()




# One image test

In [ ]:
info = next(it)
data = info['image']
label = info['label']

# Load an image and convert to RGB if necessary
input_tensor = preprocess(data).unsqueeze(0)  # Add batch dimension
input_tensor = input_tensor.to(device)

# Forward pass through the model
with torch.no_grad():
    output = model(input_tensor)

# Get predicted class
predicted_class = output.argmax(dim=-1).item()
print("ref class index:", label)
print(f"Predicted class index: {predicted_class} ")

# Map the predicted class index to the label
predicted_label = imagenet_classes[predicted_class]
actual_label = imagenet_classes[label]

print(f"Predicted label: {predicted_label} (actual: {actual_label})")

In [ ]:
# # import matplotlib.pyplot as plt
# # import numpy as np

# # # Collect variance stats from all QLayerNorm modules
# # all_vars = []
# # var_stat = np.array([])
# for m in model.modules():
#     if isinstance(m, QLayerNorm):
#         print(m.w_obs.dequantizer(m.weight_integer))
#         print(m.weight)
#         # print(m.stats['var'][-1])
#         # print(m.stats['ref'][-1])
#         print("----------------")
#         # break

In [ ]:
# asdfasdf

# memory breakdown

In [ ]:
total_weight_memory = 0

print("Weight memory breakdown:")
for name, param in model.named_parameters():
    # if param.requires_grad:
    param_memory = param.numel() * param.element_size()
    print(f"{name} ({param.dtype}): {param_memory / 1024:.2f} KB")
    total_weight_memory += param_memory

print(f"\nTotal weight memory: {total_weight_memory / 1024 / 1024:.2f} MB")


In [ ]:
parameter_activation_size = 0
parameter_activation_size_q = 0

parameter_activation_norm_size = 0
parameter_activation_norm_size_q = 0

parameter_activation_attn_size = 0
parameter_activation_attn_size_q = 0

parameter_activation_fc_size = 0
parameter_activation_fc_size_q = 0

for name, module in model.named_modules():
    # print("name:", module)
    # Check if it's a MinMaxObserver (if needed)
    # if MinMaxObserver in module.named_modules:
    # for md in module.named_modules():
    # ColorPrint.yellow(module)
    # print(MinMaxObserver, type(module), f"is instant {str(MinMaxObserver) == str(type(module))}")
    if str(MinMaxObserver) == str(type(module)):
        if module.ez:
          # ColorPrint.header(f"{name} is MinMaxObserver (nof_bits={module.nof_bits}, element_size={module.nml}, element_size={module.ez})")
          ColorPrint.header(f"{name} : shape: {module.sz} nof_bits={module.nof_bits}, size={module.nml * module.ez / 1024.0}Kb")
          ColorPrint.header(f"{name} : shape: {module.sz} nof_bits={module.nof_bits}, size={module.nml * (module.nof_bits/8) / 1024.0}Kb")
          parameter_activation_size += module.nml * module.ez / 1024.0
          parameter_activation_size_q += module.nml * (module.nof_bits/8) / 1024.0
          print("@@")

    # Now check parameters of this module
    param_size = 0
    for param in module.parameters(recurse=False):  # Only direct params (skip children)
        param_size += param.numel() * param.element_size()
        # print(f"{name} ({param.dtype}): {param_memory / 1024:.2f} KB")

print(f"parameter_activation_size: {parameter_activation_size/1024:.2f}Mb")
print(f"parameter_activation_size_q: {parameter_activation_size_q/1024:.2f}Mb")
try:
  print(f"compration ratio: {parameter_activation_size/parameter_activation_size_q*100:.2f}%")
except ZeroDivisionError:
  pass



# Model validation

In [ ]:
import time

In [ ]:
class AverageMeter(object):
    """Computes and stores the average and current value"""

    def __init__(self):
        self.reset()

    def reset(self):
        self.val = 0
        self.avg = 0
        self.sum = 0
        self.count = 0

    def update(self, val, n=1):
        self.val = val
        self.sum += val * n
        self.count += n
        self.avg = self.sum / self.count


In [ ]:
def accuracy(output, target, topk=(1, )):
    """Computes the precision@k for the specified values of k"""
    maxk = max(topk)
    batch_size = target.size(0)

    _, pred = output.topk(maxk, 1, True, True)
    pred = pred.t()
    correct = pred.eq(target.reshape(1, -1).expand_as(pred))

    res = []
    for k in topk:
        correct_k = correct[:k].reshape(-1).float().sum(0)
        res.append(correct_k.mul_(100.0 / batch_size))
    return res


In [ ]:
def validate(val_loader, model, criterion, device, samples=-1):
    batch_time = AverageMeter()
    losses = AverageMeter()
    top1 = AverageMeter()
    top5 = AverageMeter()

    # switch to evaluate mode
    model.eval()

    val_start_time = end = time.time()

    for i, info in enumerate(val_loader):

        data = torch.stack(info['image'])
        target = torch.tensor(info['label'])

        data = data.to(device)
        target = target.to(device)

        with torch.no_grad():
            output = model(data)
        loss = criterion(output, target)

        # measure accuracy and record loss
        prec1, prec5 = accuracy(output.data, target, topk=(1, 5))
        losses.update(loss.data.item(), data.size(0))
        top1.update(prec1.data.item(), data.size(0))
        top5.update(prec5.data.item(), data.size(0))

        # measure elapsed time
        batch_time.update(time.time() - end)
        end = time.time()

        # if i % args.print_freq == 0:
        if i % 100 == 0:
            print('Test: [{0}/{1}]\t'
                'Time {batch_time.val:.3f} ({batch_time.avg:.3f})\t'
                'Loss {loss.val:.4f} ({loss.avg:.4f})\t'
                'Prec@1 {top1.val:.3f} ({top1.avg:.3f})\t'
                'Prec@5 {top5.val:.3f} ({top5.avg:.3f})'.format(
                    i * batch_size,
                    50000,
                    # len(val_loader),
                    batch_time=batch_time,
                    loss=losses,
                    top1=top1,
                    top5=top5,
                ))
        if samples > 0 and i >= samples:
            print(f"validation is finished (samples {samples})")
            break
    val_end_time = time.time()
    print(' * Prec@1 {top1.avg:.3f} Prec@5 {top5.avg:.3f} Time {time:.3f}'.
            format(top1=top1, top5=top5, time=val_end_time - val_start_time))


    return losses.avg, top1.avg, top5.avg

## Inference run

In [ ]:
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

## Model inference
is_inference = 1  # TODO change to '1' whem running inference

if is_inference:
    if torch.cuda.is_available():
        device = torch.device("cuda")  # Use GPU
        print("GPU is available.")
    else:
        device = torch.device("cpu")  # Use CPU
        print("GPU is not available, using CPU.")

    try:
        criterion = nn.CrossEntropyLoss().to(device)
        print('Validating...')
        val_loss, val_prec1, val_prec5 = validate(val_loader, model,
                                                    criterion, device)
    except KeyboardInterrupt:
        print("stopped by a Keyboard Interrupt")
    finally:
        device = torch.device("cpu")  # Use CPU


In [ ]:
# DEIT T = 70.712
# DEIT S = 78.674
# DEIT B = 81.030
# SWIN T = 78.478
# SWIN S = 82.604
# Swin B = 82.794

# nlp
# sst F1: 88.7719298245614
# mnli Accuracy: 83.03
# cola MCC: 50.72753983504138
# QQP ACCURACY: 89:71
# sst-2 Accuracy: 92.6605504587156
# rte Accuracy: 71.84
# qnli Accuracy: 88.14

# LNORM STATIS

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
# Collect variance stats from all QLayerNorm modules
all_vars = []
lnorm_var_stats = dict()
lnorm_idx = 0
for m in model.modules():
    if isinstance(m, QLayerNorm):
        print("layer index:", lnorm_idx)
        lnorm_var_stats[lnorm_idx] = (m.stats['var']).detach().cpu().numpy().flatten()
        print(lnorm_var_stats[lnorm_idx].shape, m.stats['var'].shape)
        lnorm_idx +=1

# lnorm_var_stats

In [ ]:
for i in lnorm_var_stats.keys():
    print("index", i)
    print("Mean:", float(lnorm_var_stats[i].mean()) if lnorm_var_stats[i].size else "n/a")
    print("Std :", float(lnorm_var_stats[i].std()) if lnorm_var_stats[i].size else "n/a")
    print("Min :", float(lnorm_var_stats[i].min()) if lnorm_var_stats[i].size else "n/a")
    print("Max :", float(lnorm_var_stats[i].max()) if lnorm_var_stats[i].size else "n/a")
    print("---------------------------------------")

In [ ]:
# Found at: # enicskb/wiki/index.php/
# Python_-_Making_Article-Quality_Plots
# Example: Histogram of LayerNorm variance distribution

import numpy as np
import matplotlib.pyplot as plt

def plot_hist(index):
    # Extract and log2-transform variance data
    data = lnorm_var_stats[index]

    # Compute stats
    mean_val = np.mean(data)
    std_val  = np.std(data)
    min_val  = np.min(data)
    max_val  = np.max(data)

    print(f"Layer {index} stats:")
    print(f"  min  : {min_val:.4f}")
    print(f"  max  : {max_val:.4f}")
    print(f"  mean : {mean_val:.4f}")
    print(f"  std  : {std_val:.4f}")

    # Plot styling parameters
    params = {
        'legend.fontsize': '12',
        'figure.figsize': (7.5, 6.0),
        'axes.labelsize': '15',
        'axes.titlesize': '16',
        'lines.linewidth': '2',
        'xtick.labelsize': '12',
        'ytick.labelsize': '12'
    }
    plt.rcParams.update(params)

    fig, ax1 = plt.subplots()

    # Plot histogram
    ax1.hist(data, bins=20, color='bisque', edgecolor='black', alpha=0.7)

    # Labels (LaTeX style)
    ax1.set_xlabel(r'$\mathrm{LayerNorm\ Variance}$')
    ax1.set_ylabel(r'$\mathrm{Frequency}$')

    # Grid and layout
    ax1.grid(True, lw=0.5, ls='--', c='.75')

    # 🎨 Lighter, thesis-friendly line colors
    ax1.axvline(mean_val, color='#e41a1c80', linestyle='--', linewidth=2, label=fr'$\mathrm{{Mean}}={mean_val:.2f}$')
    ax1.axvline(mean_val + std_val, color='#4daf4a80', linestyle=':', linewidth=1.8, label=fr'$\mathrm{{+1\sigma}}={mean_val + std_val:.2f}$')
    ax1.axvline(mean_val - std_val, color='#4daf4a80', linestyle=':', linewidth=1.8, label=fr'$\mathrm{{-1\sigma}}={mean_val - std_val:.2f}$')
    ax1.axvline(min_val, color='#377eb880', linestyle='-.', linewidth=1.5, label=fr'$\mathrm{{Min}}={min_val:.2f}$')
    ax1.axvline(max_val, color='#984ea380', linestyle='-.', linewidth=1.5, label=fr'$\mathrm{{Max}}={max_val:.2f}$')

    # Optional: place legend nicely
    ax1.legend(loc='upper right', fontsize=11, frameon=False)

    fig.tight_layout()

    # Save high-quality PDF
    plt.savefig(f"lnorm_variance_distribution_layer{index}_{model_name}.pdf")

    plt.show()


In [ ]:
# Found at: # enicskb/wiki/index.php/
# Python_-_Making_Article-Quality_Plots
# Example: Histogram of LayerNorm variance distribution

import numpy as np
import matplotlib.pyplot as plt

def plot_log2_hist(index):
    # Extract and log2-transform variance data
    data = np.log2(lnorm_var_stats[index])

    # Compute stats
    mean_val = np.mean(data)
    std_val  = np.std(data)
    min_val  = np.min(data)
    max_val  = np.max(data)

    print(f"Layer {index} stats:")
    print(f"  min  : {min_val:.4f}")
    print(f"  max  : {max_val:.4f}")
    print(f"  mean : {mean_val:.4f}")
    print(f"  std  : {std_val:.4f}")

    # Plot styling parameters
    params = {
        'legend.fontsize': '12',
        'figure.figsize': (7.5, 6.0),
        'axes.labelsize': '15',
        'axes.titlesize': '16',
        'lines.linewidth': '2',
        'xtick.labelsize': '12',
        'ytick.labelsize': '12'
    }
    plt.rcParams.update(params)

    fig, ax1 = plt.subplots()

    # Plot histogram
    ax1.hist(data, bins=20, color='bisque', edgecolor='black', alpha=0.7)

    # Labels (LaTeX style)
    ax1.set_xlabel(r'$\log_2(\mathrm{LayerNorm\ Variance})$')
    ax1.set_ylabel(r'$\mathrm{Frequency}$')

    # Grid and layout
    ax1.grid(True, lw=0.5, ls='--', c='.75')

    # 🎨 Lighter, thesis-friendly line colors
    ax1.axvline(mean_val, color='#e41a1c80', linestyle='--', linewidth=2, label=fr'$\mathrm{{Mean}}={mean_val:.2f}$')
    ax1.axvline(mean_val + std_val, color='#4daf4a80', linestyle=':', linewidth=1.8, label=fr'$\mathrm{{+1\sigma}}={mean_val + std_val:.2f}$')
    ax1.axvline(mean_val - std_val, color='#4daf4a80', linestyle=':', linewidth=1.8, label=fr'$\mathrm{{-1\sigma}}={mean_val - std_val:.2f}$')
    ax1.axvline(min_val, color='#377eb880', linestyle='-.', linewidth=1.5, label=fr'$\mathrm{{Min}}={min_val:.2f}$')
    ax1.axvline(max_val, color='#984ea380', linestyle='-.', linewidth=1.5, label=fr'$\mathrm{{Max}}={max_val:.2f}$')

    # Optional: place legend nicely
    ax1.legend(loc='upper right', fontsize=11, frameon=False)

    fig.tight_layout()

    # Save high-quality PDF
    plt.savefig(f"lnorm_log2_variance_distribution_layer{index}_{model_name}.pdf")

    plt.show()


In [ ]:
# Found at: # enicskb/wiki/index.php/
# Python_-_Making_Article-Quality_Plots
# Example: Histogram of LayerNorm variance distribution

import numpy as np
import matplotlib.pyplot as plt

def plot_log2_hist(index):
    # Extract and log2-transform variance data
    data = np.log2(lnorm_var_stats[index])

    # Compute stats
    mean_val = np.mean(data)
    std_val  = np.std(data)
    min_val  = np.min(data)
    max_val  = np.max(data)

    print(f"Layer {index} stats:")
    print(f"  min  : {min_val:.4f}")
    print(f"  max  : {max_val:.4f}")
    print(f"  mean : {mean_val:.4f}")
    print(f"  std  : {std_val:.4f}")

    # Plot styling parameters
    params = {
        'legend.fontsize': '12',
        'figure.figsize': (7.5, 6.0),
        'axes.labelsize': '15',
        'axes.titlesize': '16',
        'lines.linewidth': '2',
        'xtick.labelsize': '12',
        'ytick.labelsize': '12'
    }
    plt.rcParams.update(params)

    fig, ax1 = plt.subplots()

    # Plot histogram
    ax1.hist(data, bins=20, color='bisque', edgecolor='black', alpha=0.7)

    # Labels (LaTeX style)
    ax1.set_xlabel(r'$\log_2(\mathrm{LayerNorm\ Variance})$')
    ax1.set_ylabel(r'$\mathrm{Frequency}$')

    # Grid and layout
    ax1.grid(True, lw=0.5, ls='--', c='.75')

    # 🎨 Lighter, thesis-friendly line colors
    ax1.axvline(mean_val, color='#e41a1c80', linestyle='--', linewidth=2, label=fr'$\mathrm{{Mean}}={mean_val:.2f}$')
    ax1.axvline(mean_val + std_val, color='#4daf4a80', linestyle=':', linewidth=1.8, label=fr'$\mathrm{{+1\sigma}}={mean_val + std_val:.2f}$')
    ax1.axvline(mean_val - std_val, color='#4daf4a80', linestyle=':', linewidth=1.8, label=fr'$\mathrm{{-1\sigma}}={mean_val - std_val:.2f}$')
    ax1.axvline(min_val, color='#377eb880', linestyle='-.', linewidth=1.5, label=fr'$\mathrm{{Min}}={min_val:.2f}$')
    ax1.axvline(max_val, color='#984ea380', linestyle='-.', linewidth=1.5, label=fr'$\mathrm{{Max}}={max_val:.2f}$')

    # Optional: place legend nicely
    ax1.legend(loc='upper right', fontsize=11, frameon=False)

    fig.tight_layout()

    # Save high-quality PDF
    plt.savefig(f"lnorm_log2_variance_distribution_layer{model_name}.pdf")

    plt.show()



In [ ]:
# Found at: # enicskb/wiki/index.php/
# Python_-_Making_Article-Quality_Plots
# Example: Multi-layer Vertical Boxplot of LayerNorm variance distribution

import numpy as np
import matplotlib.pyplot as plt

def plot_lnorm_boxplot_layers(index_list):
    # Prepare data for each requested layer
    all_data = [np.log2(lnorm_var_stats[idx]) for idx in index_list]

    # Compute and print statistics for each layer
    print("LayerNorm variance stats (log2 scale):")
    for idx, data in zip(index_list, all_data):
        mean_val = np.mean(data)
        std_val  = np.std(data)
        min_val  = np.min(data)
        max_val  = np.max(data)
        print(f"Layer {idx}: min={min_val:.4f}, max={max_val:.4f}, mean={mean_val:.4f}, std={std_val:.4f}")

    # Plot styling parameters
    params = {
        'legend.fontsize': '12',
        'figure.figsize': (8, 5),
        'axes.labelsize': '15',
        'axes.titlesize': '16',
        'lines.linewidth': '2',
        'xtick.labelsize': '12',
        'ytick.labelsize': '12'
    }
    plt.rcParams.update(params)

    fig, ax1 = plt.subplots()

    # 📊 Vertical boxplot (rotated orientation)
    bp = ax1.boxplot(
        all_data,
        vert=True,
        showfliers=True,
        patch_artist=True,
        boxprops=dict(facecolor='bisque', color='black', alpha=0.7),
        medianprops=dict(color='darkred', linewidth=2),
        whiskerprops=dict(color='black', linewidth=1.5),
        capprops=dict(color='black', linewidth=1.5),
        flierprops=dict(marker='o', color='gray', alpha=0.5)
    )

    # X-axis ticks: layer labels
    ax1.set_xticks(np.arange(1, len(index_list) + 1))
    ax1.set_xticklabels([f"{idx}" for idx in index_list], rotation=90, ha='right')

    # Labels
    ax1.set_ylabel(r'$\log_2(\mathrm{LayerNorm\ Variance})$')
    ax1.set_xlabel(r'$\mathrm{Layer}$')
    # ax1.set_title(r'$\mathrm{Variance\ Distribution\ across\ QLayerNorm\ Layers}$')

    # Grid and layout
    ax1.grid(True, lw=0.5, ls='--', c='.75', axis='y')
    fig.tight_layout()

    # Save as high-quality PDF
    layer_str = "_".join(map(str, index_list))
    plt.savefig(f"lnorm_variance_boxplot_layers_{model_name}_vertical.pdf")

    plt.show()


In [ ]:
plot_hist(47)
plot_log2_hist(47)
plot_lnorm_boxplot_layers([i for i in lnorm_var_stats.keys()])

In [ ]:
plt.figure(figsize=(8,5))
plt.hist(soft_stat, bins=20, color="darkorange", edgecolor="black", alpha=0.7)
plt.xlabel("lnorm variance")
plt.ylabel("Frequency")
plt.title("Distribution of variance")
plt.grid(True, linestyle="--", alpha=0.6)
plt.show()

plt.figure(figsize=(8,5))
plt.hist(np.log2(soft_stat), bins=20, color="darkorange", edgecolor="black", alpha=0.7)
plt.xlabel("lnorm variance")
plt.ylabel("Frequency")
plt.title("Distribution of variance (log-scaled)")
plt.grid(True, linestyle="--", alpha=0.6)
plt.show()

plt.figure(figsize=(6,4))
plt.boxplot(soft_stat, vert=False, showfliers=True)
plt.xscale("log")
plt.xlabel("Variance (log scale)")
plt.title("Boxplot of variance across QLayerNorm")
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Collect variance stats from all QLayerNorm modules
all_vars = []
var_stat = np.array([])
for m in model.modules():
    if isinstance(m, QLayerNorm):
        for v in m.stats['var']:
            v = v.detach().cpu().numpy().flatten()
            var_stat = np.concatenate((var_stat, v))

        if len(var_stat) > 19000000:
            break
print(len(var_stat))


In [ ]:

print("Mean:", float(var_stat.mean()) if var_stat.size else "n/a")
print("Std :", float(var_stat.std()) if var_stat.size else "n/a")
print("Min :", float(var_stat.min()) if var_stat.size else "n/a")
print("Max :", float(var_stat.max()) if var_stat.size else "n/a")


In [ ]:
plt.figure(figsize=(8,5))
plt.hist(var_stat, bins=20, color="darkorange", edgecolor="black", alpha=0.7)
plt.xlabel("lnorm variance")
plt.ylabel("Frequency")
plt.title("Distribution of variance")
plt.grid(True, linestyle="--", alpha=0.6)
plt.show()

plt.figure(figsize=(8,5))
plt.hist(np.log2(var_stat), bins=20, color="darkorange", edgecolor="black", alpha=0.7)
plt.xlabel("lnorm variance")
plt.ylabel("Frequency")
plt.title("Distribution of variance (log-scaled)")
plt.grid(True, linestyle="--", alpha=0.6)
plt.show()


In [ ]:
log_vars = np.log10(all_vars + 1e-12)  # avoid log(0)
plt.figure(figsize=(8,5))
plt.hist(log_vars, bins=20, color="darkorange", edgecolor="black", alpha=0.7)
plt.xlabel("log10(variance)")
plt.ylabel("Frequency")
plt.title("Distribution of variance (log-scaled)")
plt.grid(True, linestyle="--", alpha=0.6)
plt.show()

log_vars = np.log2(all_vars + 1e-12)  # avoid log(0)
plt.figure(figsize=(8,5))
plt.hist(log_vars, bins=20, color="darkorange", edgecolor="black", alpha=0.7)
plt.xlabel("log2(variance)")
plt.ylabel("Frequency")
plt.title("Distribution of variance (log-scaled)")
plt.grid(True, linestyle="--", alpha=0.6)
plt.show()

log_vars = np.log(all_vars + 1e-12)  # avoid log(0)
plt.figure(figsize=(8,5))
plt.hist(log_vars, bins=20, color="darkorange", edgecolor="black", alpha=0.7)
plt.xlabel("log(variance)")
plt.ylabel("Frequency")
plt.title("Distribution of variance (log-scaled)")
plt.grid(True, linestyle="--", alpha=0.6)
plt.show()

In [ ]:
plt.figure(figsize=(6,4))
plt.boxplot(all_vars, vert=False, showfliers=True)
plt.xscale("log")
plt.xlabel("Variance (log scale)")
plt.title("Boxplot of variance across QLayerNorm")
plt.show()

In [ ]:
sorted_vars = np.sort(all_vars)
cdf = np.arange(len(sorted_vars)) / float(len(sorted_vars)-1)
plt.figure(figsize=(6,4))
plt.plot(sorted_vars, cdf)
plt.xscale("log")
plt.xlabel("Variance")
plt.ylabel("Cumulative Probability")
plt.title("CDF of variance across QLayerNorm")
plt.grid(True, linestyle="--", alpha=0.6)
plt.show()

In [ ]:
import numpy as np
import torch

def _to_np_1d(x):
    if isinstance(x, torch.Tensor):
        x = x.detach().cpu().numpy()
    return np.asarray(x).ravel()

def _flatten_with_index(all_vars):
    """Return flat values and (series_idx, elem_idx) mapping."""
    if isinstance(all_vars, (list, tuple)):
        vals = []
        map_idx = []
        for si, arr in enumerate(all_vars):
            a = _to_np_1d(arr)
            vals.append(a)
            map_idx.extend([(si, j) for j in range(len(a))])
        vals = np.concatenate(vals) if len(vals) else np.array([])
        map_idx = np.asarray(map_idx, dtype=int) if len(map_idx) else np.zeros((0,2), dtype=int)
        return vals, map_idx
    else:
        a = _to_np_1d(all_vars)
        map_idx = np.column_stack([np.zeros(len(a), dtype=int), np.arange(len(a))])
        return a, map_idx

def find_outliers(all_vars, method="iqr", tail="both", use_log=False,
                  k=1.5, z=3.0, mad_z=3.5, eps=1e-12):
    """
    Detect outliers in all_vars.
      method: "iqr" | "z" | "mad"
      tail  : "both" | "high" | "low"
      use_log: apply detection on log(values+eps)
      k     : IQR multiplier (1.5 = Tukey)
      z     : z-score threshold
      mad_z : robust z using MAD threshold (~3.5 common)
    Returns dict with mask, thresholds and indexed outliers.
    """
    x, map_idx = _flatten_with_index(all_vars)
    x = x.astype(float, copy=False)
    x_work = np.log(x + eps) if use_log else x

    lo_thr, hi_thr = -np.inf, np.inf

    if method == "iqr":
        q1, q3 = np.percentile(x_work, [25, 75])
        iqr = q3 - q1
        lo_thr = q1 - k * iqr
        hi_thr = q3 + k * iqr
        mask_low  = x_work < lo_thr
        mask_high = x_work > hi_thr
    elif method == "z":
        m, s = x_work.mean(), x_work.std(ddof=0)
        s = max(s, 1e-30)
        zscores = (x_work - m) / s
        mask_low  = zscores < -z
        mask_high = zscores >  z
    elif method == "mad":
        med = np.median(x_work)
        mad = np.median(np.abs(x_work - med))
        mad = max(mad, 1e-30)
        robust_z = 0.67448975 * (x_work - med) / mad  # ~Phi^-1(0.75) scaling
        mask_low  = robust_z < -mad_z
        mask_high = robust_z >  mad_z
    else:
        raise ValueError("method must be one of: 'iqr', 'z', 'mad'")

    if tail == "both":
        mask = mask_low | mask_high
    elif tail == "high":
        mask = mask_high
        lo_thr = -np.inf  # not used
    elif tail == "low":
        mask = mask_low
        hi_thr =  np.inf  # not used
    else:
        raise ValueError("tail must be 'both', 'high', or 'low'")

    idx = np.where(mask)[0]
    out = {
        "count": int(mask.sum()),
        "mask": mask,
        "indices_flat": idx,
        "indices_series_elem": map_idx[idx],   # (series_idx, element_idx)
        "values": x[idx],
        "thresholds": {"low": float(lo_thr), "high": float(hi_thr)},
        "used_log": bool(use_log),
        "method": method,
        "tail": tail,
    }
    return out

# --------------------
# Usage examples
# --------------------
# 1) Simple: treat all values together, catch high-end outliers with Tukey on log-scale
# (good for variances)
res = find_outliers(all_vars, method="iqr", tail="high", use_log=True, k=1.5)

print(f"Outliers found: {res['count']}")
print("Thresholds (on transformed axis):", res["thresholds"])
# Show top 10 biggest outliers (by raw value)
order = np.argsort(res["values"])[::-1]
for n in order[:10]:
    si, ei = res["indices_series_elem"][n]
    val = res["values"][n]
    print(f"series={si}, idx={ei}, value={val:.6g}")

# 2) Robust alternative:
res = find_outliers(all_vars, method="mad", tail="high", use_log=True, mad_z=3.5)
print(res)
# 3) If you want to see both tails without log:
res = find_outliers(all_vars, method="iqr", tail="both", use_log=False, k=1.5)
print(res)

# gelu distrebution

In [ ]:
import torch

# Step 1: Precompute the table
x_table = torch.linspace(-5, 5, steps=1000)
normal = torch.distributions.Normal(0.0, 1.0)
cdf_table = normal.cdf(x_table)

def approx_cdf(x_query: torch.Tensor, x_table: torch.Tensor, cdf_table: torch.Tensor):
    # Clamp the input range
    x_query = x_query.clamp(min=x_table[0].item(), max=x_table[-1].item())

    # Scale to table index
    idx = (x_query - x_table[0]) / (x_table[1] - x_table[0])
    idx_low = torch.floor(idx).long()
    idx_high = torch.clamp(idx_low + 1, max=len(x_table) - 1)

    # Linear interpolation
    weight = idx - idx_low
    cdf_approx = (1 - weight) * cdf_table[idx_low] + weight * cdf_table[idx_high]

    return cdf_approx


In [ ]:
import torch
import matplotlib.pyplot as plt

# Create a normal distribution
mu = 0.0
sigma = 1.0
normal = torch.distributions.Normal(mu, sigma)

# x values
x = torch.linspace(-5, 5, steps=1000)
# CDF values
cdf = normal.cdf(x)
apprx = torch.sigmoid(1.752*x)
# Plot
plt.plot(x.numpy(), cdf.numpy(), label='CDF')
plt.plot(x.numpy(), apprx.numpy(), label='SIG')
plt.title('CDF of Normal Distribution (μ=0, σ=1)')
plt.xlabel('x')
plt.ylabel('CDF(x)')
plt.grid(True)
plt.legend()
plt.show()


In [ ]:
import torch
import matplotlib.pyplot as plt

# -------------------------------------------------
# Data
# -------------------------------------------------
mu = 0.0
sigma = 1.0
normal = torch.distributions.Normal(mu, sigma)

x_axis = torch.linspace(-4, 4, steps=1000)
xaxis_np = x_axis.numpy()

cdf = normal.cdf(x_axis).numpy()
sig = torch.sigmoid(1.752 * x_axis).numpy()

plot_div = {
    "Normal CDF": cdf,
    "Sigmoid (1.752·x)": sig
}

argVecs  = []
argNames = []
for label, data in plot_div.items():
    argVecs.append(data)
    argNames.append(label)

# -------------------------------------------------
# Matplotlib parameters (from your code)
# -------------------------------------------------
params = {
    'legend.fontsize': '12',
    'figure.figsize': (7.5, 6.0),
    'axes.labelsize': '15',
    'axes.titlesize': '14',
    'lines.linewidth': '2',
    'xtick.labelsize': '12',
    'ytick.labelsize': '12'
}
plt.rcParams.update(params)

colors = [
    '#0072B2',  # Dark Blue
    '#D55E00',  # Orange-Red
    '#009E73',  # Vivid Green
    '#CC79A7',  # Rosy Purple
]

# -------------------------------------------------
# Plot
# -------------------------------------------------
fig, ax1 = plt.subplots(constrained_layout=True)

for i in range(len(argVecs)):
    ax1.plot(
        xaxis_np,
        argVecs[i],
        label=argNames[i],
        color=colors[i % len(colors)],
        linewidth=1,
        marker='o',
        markersize=2
    )

ax1.set_xlabel('x')
ax1.set_ylabel('CDF / Sigmoid', color='blue')
ax1.tick_params('y', colors='blue')


ax1.legend(loc='upper left')
ax1.grid(True, lw=0.5, ls='--', c='.75')

plt.savefig(f"CDF_VS_SIGMOID.pdf")
plt.show()


In [ ]:
import torch
import matplotlib.pyplot as plt

# Create a normal distribution
mu = 0.0
sigma = 1.0
normal = torch.distributions.Normal(mu, sigma)

for scale in [1.670, 1.702, 1.752, 1.802]:
  # x values
  x = torch.linspace(-4, 4, steps=1000)
  # CDF values
  cdf = x*normal.cdf(x)
  apprx = x*torch.sigmoid(scale*x)
  # Plot
  plt.plot(x.numpy(), cdf.numpy(), label='CDF')
  plt.plot(x.numpy(), apprx.numpy(), label=f'SIG({scale})')
  plt.title('CDF of Normal Distribution (μ=0, σ=1)')
  plt.xlabel('x')
  plt.ylabel('CDF(x)')
  plt.grid(True)
  plt.legend()
  plt.show()


In [ ]:
import torch

# Step 1: Precompute the table
x_table = torch.linspace(-5, 5, steps=1000)
normal = torch.distributions.Normal(0.0, 1.0)
cdf_table = normal.cdf(x_table)

def approx_cdf(x_query: torch.Tensor, x_table: torch.Tensor, cdf_table: torch.Tensor):
    # Clamp the input range
    x_query = x_query.clamp(min=x_table[0].item(), max=x_table[-1].item())

    # Scale to table index
    idx = (x_query - x_table[0]) / (x_table[1] - x_table[0])
    idx_low = torch.floor(idx).long()
    idx_high = torch.clamp(idx_low + 1, max=len(x_table) - 1)

    # Linear interpolation
    weight = idx - idx_low
    cdf_approx = (1 - weight) * cdf_table[idx_low] + weight * cdf_table[idx_high]

    return cdf_approx


In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize_scalar

# Exact GELU function
def gelu(x):
    return 0.5 * x * (1 + torch.tanh(torch.sqrt(torch.tensor(2 / np.pi)) * (x + 0.044715 * x**3)))

# Approximate GELU using sigmoid
def approx_gelu(x, k):
    return x * torch.sigmoid(k * x)

# Define input regions
regions = [(5, 4), (4, 3), (3, 2), (2, 1), (1, 0), (0, -1), (-1, -2), (-2, -3), (-3, -4), (-4, -5)]
results = []

# Plotting setup
fig, axes = plt.subplots(len(regions) // 2, 2, figsize=(12, 18))
axes = axes.flatten()

# Process each region
for i, (upper, lower) in enumerate(regions):
    x_vals = torch.linspace(lower, upper, steps=1000)
    gelu_true = gelu(x_vals)

    def loss(k):
        return torch.mean((gelu_true - approx_gelu(x_vals, k)) ** 2).item()

    res = minimize_scalar(loss, bounds=(0.5, 3.0), method='bounded')
    best_k = res.x
    best_mse = res.fun
    gelu_approx = approx_gelu(x_vals, best_k)

    results.append((f"{upper}:{lower}", best_k, best_mse))

    ax = axes[i]
    ax.plot(x_vals.numpy(), gelu_true.numpy(), label='True GELU', linewidth=2)
    ax.plot(x_vals.numpy(), gelu_approx.detach().numpy(), '--', label=f'Approx (k={best_k:.4f})')
    ax.set_title(f"Region {upper}:{lower} | MSE: {best_mse:.2e}")
    ax.legend()
    ax.grid(True)

plt.tight_layout()
plt.show()

# Print table of results
print("\nBest k per region:")
for region, k, mse in results:
    print(f"Region {region} → Best k: {k:.6f}, MSE: {mse:.6e}")


In [ ]:
!pip install onnx

In [ ]:
info = next(it)
data = info['image']
label = info['label']

# Load an image and convert to RGB if necessary
input_tensor = preprocess(data).unsqueeze(0)  # Add batch dimension
dummy_input = input_tensor.to("cuda")

# 3. Export the model to ONNX
onnx_file_path = "simplenet.onnx"
torch.onnx.export(
    model,
    dummy_input,
    onnx_file_path,
    export_params=True,
    opset_version=17,
    do_constant_folding=True,
    input_names=['input'],
    output_names=['output'],
    dynamic_axes={
        'input': {0: 'batch_size'},
        'output': {0: 'batch_size'}
    }
)

print(f"Model successfully converted to ONNX and saved to {onnx_file_path}")

# (Optional) Verify the exported ONNX model
try:
    import onnx
    onnx_model = onnx.load(onnx_file_path)
    onnx.checker.check_model(onnx_model)
    print("ONNX model verification passed!")
except ImportError:
    print("The 'onnx' package is not installed. Skipping model verification.")
except onnx.checker.ValidationError as e:
    print(f"ONNX model verification failed: {e}")
